# Portfolio Robustness and Implementation

## Objective

This notebook tests whether the shortlisted portfolio constructions remain viable under alternative implementation assumptions.

The primary portfolios are:

1. Composite Score;
2. Fixed 50/50 Sleeves;
3. Pure Inverse Volatility; and
4. SPY Buy and Hold as the passive benchmark.

The first experiment jointly evaluates:

- transaction costs of 0, 5, 10, 20, and 50 basis points;
- rebalancing every 1, 5, 10, and 21 trading days;
- turnover, net performance, drawdown, and exposure; and
- performance over identical evaluation dates.

The original specification—rebalancing every five trading days with 10-bps transaction costs—is retained as the baseline rather than reselected from the robustness results.

## Design principles

- Reconstruct security-level target weights for every rebalance schedule.
- Preserve the original factor definitions, quantile construction, exposure, and sign conventions.
- Estimate inverse-volatility allocations using lagged sleeve returns only.
- Apply the same drift-aware turnover and transaction-cost model used previously.
- Treat the cost and frequency grids as pre-specified sensitivity tests, not parameters to optimise.
- Compare all variants over a common evaluation window.
- Test rebalance-offset sensitivity separately before interpreting any preferred non-daily frequency.
- Treat SPY as buy-and-hold: its holdings do not change with the active portfolios' rebalance frequency.

In [1]:
import numpy as np
import pandas as pd

from alpha_research.backtest import (
    BacktestConfig,
    run_target_weight_backtest,
)
from alpha_research.config.paths import PROCESSED_DATA_DIR
from alpha_research.data_loader import load_parquet
from alpha_research.portfolio import (
    build_factor_target_weights,
    calculate_inverse_volatility_allocations,
    combine_dynamic_sleeve_target_weights,
    combine_factor_scores,
    combine_sleeve_target_weights,
    estimate_trailing_sleeve_volatility,
)

FACTOR_COLUMNS = {
    "Momentum": "mom_12_1m_z",
    "Realised Volatility": "realised_vol_63_z",
}

COMPOSITE_COLUMN = "mom_vol_composite_z"

BASELINE_REBALANCE_FREQUENCY = 5
BASELINE_TRANSACTION_COST_BPS = 10.0

REBALANCE_FREQUENCIES = [1, 5, 10, 21]
TRANSACTION_COST_GRID_BPS = [0.0, 5.0, 10.0, 20.0, 50.0]

ACTIVE_PORTFOLIOS = [
    "Composite Score",
    "Fixed 50/50 Sleeves",
    "Pure Inverse Volatility",
]

EXPECTED_BENCHMARK_PORTFOLIOS = {
    "Composite Score",
    "Fixed 50/50 Sleeves",
    "Shrunk Inverse Volatility",
    "Pure Inverse Volatility",
    "SPY Buy and Hold",
}

panel = load_parquet(PROCESSED_DATA_DIR / "factor_panel.parquet")

exported_benchmarks = load_parquet(
    PROCESSED_DATA_DIR / "portfolio_optimisation_benchmarks.parquet"
)

panel["date"] = pd.to_datetime(panel["date"])
exported_benchmarks["date"] = pd.to_datetime(exported_benchmarks["date"])

required_panel_columns = {
    "date",
    "ticker",
    "forward_ret_1d",
    *FACTOR_COLUMNS.values(),
}

missing_panel_columns = required_panel_columns - set(panel.columns)

if missing_panel_columns:
    raise ValueError(
        "Missing factor-panel columns: " f"{sorted(missing_panel_columns)}"
    )

if panel.duplicated(["date", "ticker"]).any():
    raise ValueError("Duplicate date-ticker rows found in factor panel.")

if exported_benchmarks.duplicated(["portfolio", "date"]).any():
    raise ValueError("Duplicate portfolio-date rows found in benchmarks.")

available_benchmarks = set(exported_benchmarks["portfolio"].unique())

if available_benchmarks != EXPECTED_BENCHMARK_PORTFOLIOS:
    raise ValueError(
        "Unexpected benchmark portfolios: " f"{sorted(available_benchmarks)}"
    )

return_panel = panel[["date", "ticker", "forward_ret_1d"]].copy()

composite_panel = panel.copy()

composite_panel[COMPOSITE_COLUMN] = combine_factor_scores(
    panel=composite_panel,
    factor_weights={
        "mom_12_1m_z": 0.5,
        "realised_vol_63_z": 0.5,
    },
)

In [2]:
input_audit = pd.DataFrame(
    {
        "dataset": [
            "Factor panel",
            "Exported benchmark portfolios",
        ],
        "rows": [
            len(panel),
            len(exported_benchmarks),
        ],
        "start_date": [
            panel["date"].min(),
            exported_benchmarks["date"].min(),
        ],
        "end_date": [
            panel["date"].max(),
            exported_benchmarks["date"].max(),
        ],
    }
)

benchmark_audit = (
    exported_benchmarks.groupby("portfolio")
    .agg(
        observations=("date", "size"),
        start_date=("date", "min"),
        end_date=("date", "max"),
    )
    .sort_index()
)

display(input_audit)
display(benchmark_audit)

,dataset,rows,start_date,end_date
0,Factor panel,284249,2015-01-02,2026-07-02
1,Exported benchmark portfolios,13175,2016-01-07,2026-07-01


,observations,start_date,end_date
portfolio,,,
Composite Score,2635,2016-01-07,2026-07-01
Fixed 50/50 Sleeves,2635,2016-01-07,2026-07-01
Pure Inverse Volatility,2635,2016-01-07,2026-07-01
SPY Buy and Hold,2635,2016-01-07,2026-07-01
Shrunk Inverse Volatility,2635,2016-01-07,2026-07-01


#### Reusable portfolio-construction function

In [3]:
def extract_active_gross_returns(
    daily: pd.DataFrame,
    exposure_tolerance: float = 1e-12,
) -> pd.Series:
    indexed = (
        daily.copy()
        .assign(date=lambda df: pd.to_datetime(df["date"]))
        .set_index("date")
        .sort_index()
    )

    return indexed["gross_return"].where(indexed["gross_exposure"] > exposure_tolerance)


def calculate_rebalance_inverse_volatility_allocations(
    sleeve_returns: pd.DataFrame,
    rebalance_dates: pd.DatetimeIndex,
    lookback: int = 63,
    min_periods: int = 42,
    periods_per_year: int = 252,
) -> pd.DataFrame:
    """Calculate lagged pure inverse-volatility allocations.

    Each allocation uses the most recent jointly available sleeve
    returns strictly before its rebalance date. Equal weights are used
    during the estimation warm-up period.
    """
    returns = sleeve_returns.copy().sort_index()

    if not isinstance(returns.index, pd.DatetimeIndex):
        raise ValueError("sleeve_returns must have a DatetimeIndex.")

    if returns.index.duplicated().any():
        raise ValueError("sleeve_returns contains duplicate dates.")

    rebalance_dates = pd.DatetimeIndex(rebalance_dates).sort_values().unique()

    allocations = pd.DataFrame(
        1.0 / returns.shape[1],
        index=rebalance_dates,
        columns=returns.columns,
        dtype=float,
    )

    allocations.index.name = "date"

    for date in rebalance_dates:
        history = returns.loc[returns.index < date].dropna(how="any").tail(lookback)

        if len(history) < min_periods:
            continue

        volatility = history.std(ddof=1) * np.sqrt(periods_per_year)

        if (
            volatility.isna().any()
            or not np.isfinite(volatility.to_numpy()).all()
            or (volatility <= 0.0).any()
        ):
            continue

        inverse_volatility = 1.0 / volatility

        allocations.loc[date] = inverse_volatility / inverse_volatility.sum()

    if not np.allclose(
        allocations.sum(axis=1),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):
        raise ValueError("Inverse-volatility allocations do not sum to one.")

    return allocations


def build_robustness_portfolios(
    rebalance_frequency: int,
    rebalance_offset: int = 0,
    transaction_cost_bps: float = (BASELINE_TRANSACTION_COST_BPS),
) -> dict[str, object]:
    config = BacktestConfig(
        rebalance_frequency=rebalance_frequency,
        quantiles=5,
        long_quantile=5,
        short_quantile=1,
        long_gross=1.0,
        short_gross=1.0,
        transaction_cost_bps=transaction_cost_bps,
        min_observations=30,
        rebalance_offset=rebalance_offset,
    )

    sleeve_targets = {
        sleeve_name: build_factor_target_weights(
            panel=panel,
            factor_column=factor_column,
            return_column="forward_ret_1d",
            config=config,
        )
        for sleeve_name, factor_column in FACTOR_COLUMNS.items()
    }

    target_date_indexes = {
        sleeve_name: pd.DatetimeIndex(targets["date"].unique()).sort_values()
        for sleeve_name, targets in sleeve_targets.items()
    }

    reference_target_dates = target_date_indexes["Momentum"]

    for sleeve_name, dates in target_date_indexes.items():
        if not dates.equals(reference_target_dates):
            raise ValueError(
                f"{sleeve_name} target dates do not " "match momentum target dates."
            )

    sleeve_daily = {}

    for sleeve_name, targets in sleeve_targets.items():
        daily, _ = run_target_weight_backtest(
            return_panel=return_panel,
            target_weights=targets,
            return_column="forward_ret_1d",
            transaction_cost_bps=0.0,
        )

        sleeve_daily[sleeve_name] = daily

    sleeve_return_frame = pd.concat(
        {
            sleeve_name: extract_active_gross_returns(daily)
            for sleeve_name, daily in sleeve_daily.items()
        },
        axis=1,
    ).sort_index()

    pure_allocations = (
        calculate_rebalance_inverse_volatility_allocations(
            sleeve_returns=sleeve_return_frame,
            rebalance_dates=reference_target_dates,
            lookback=63,
            min_periods=42,
            periods_per_year=252,
        )
    )

    pure_allocations.index.name = "date"

    if pure_allocations.isna().any().any():
        raise ValueError(
            "Pure inverse-volatility allocations " "contain missing values."
        )

    if not np.allclose(
        pure_allocations.sum(axis=1),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ):
        raise ValueError("Pure inverse-volatility allocations " "do not sum to one.")

    fixed_targets = combine_sleeve_target_weights(
        sleeve_targets=sleeve_targets,
        sleeve_allocations={
            "Momentum": 0.5,
            "Realised Volatility": 0.5,
        },
    )

    pure_targets = combine_dynamic_sleeve_target_weights(
        sleeve_targets=sleeve_targets,
        sleeve_allocations=pure_allocations,
    )

    composite_targets = build_factor_target_weights(
        panel=composite_panel,
        factor_column=COMPOSITE_COLUMN,
        return_column="forward_ret_1d",
        config=config,
    )

    portfolio_targets = {
        "Composite Score": composite_targets,
        "Fixed 50/50 Sleeves": fixed_targets,
        "Pure Inverse Volatility": pure_targets,
    }

    portfolio_daily = {}
    portfolio_trades = {}

    for portfolio_name, targets in portfolio_targets.items():
        daily, holdings = run_target_weight_backtest(
            return_panel=return_panel,
            target_weights=targets,
            return_column="forward_ret_1d",
            transaction_cost_bps=transaction_cost_bps,
        )

        daily["date"] = pd.to_datetime(daily["date"])
        holdings["date"] = pd.to_datetime(holdings["date"])

        # Retain only actual security-level orders.
        trades = holdings.loc[
            holdings["trade"].ne(0.0),
            [
                "date",
                "ticker",
                "trade",
            ],
        ].copy()

        portfolio_daily[portfolio_name] = daily
        portfolio_trades[portfolio_name] = trades

    return {
        "daily": portfolio_daily,
        "trades": portfolio_trades,
        "targets": portfolio_targets,
        "sleeve_daily": sleeve_daily,
        "pure_allocations": pure_allocations,
    }

#### Reproduce baseline results

In [4]:
baseline_reconstruction = build_robustness_portfolios(
    rebalance_frequency=(BASELINE_REBALANCE_FREQUENCY),
    rebalance_offset=0,
    transaction_cost_bps=(BASELINE_TRANSACTION_COST_BPS),
)

BASELINE_AUDIT_COLUMNS = [
    "gross_return",
    "net_return",
    "turnover",
    "transaction_cost",
    "missing_return_weight",
    "gross_exposure",
    "net_exposure",
]

baseline_audit_rows = []

for portfolio_name in ACTIVE_PORTFOLIOS:
    reconstructed = baseline_reconstruction["daily"][portfolio_name][
        [
            "date",
            "is_rebalance",
            *BASELINE_AUDIT_COLUMNS,
        ]
    ].copy()

    exported = exported_benchmarks.loc[
        exported_benchmarks["portfolio"] == portfolio_name,
        [
            "date",
            "is_rebalance",
            *BASELINE_AUDIT_COLUMNS,
        ],
    ].copy()

    aligned = reconstructed.merge(
        exported,
        on="date",
        how="inner",
        suffixes=("_reconstructed", "_exported"),
        validate="one_to_one",
    )

    maximum_absolute_difference = max(
        (aligned[f"{column}_reconstructed"] - aligned[f"{column}_exported"]).abs().max()
        for column in BASELINE_AUDIT_COLUMNS
    )

    rebalance_flags_match = (
        aligned["is_rebalance_reconstructed"].eq(aligned["is_rebalance_exported"]).all()
    )

    baseline_audit_rows.append(
        {
            "portfolio": portfolio_name,
            "overlapping_observations": len(aligned),
            "start_date": aligned["date"].min(),
            "end_date": aligned["date"].max(),
            "maximum_absolute_difference": (maximum_absolute_difference),
            "rebalance_flags_match": (rebalance_flags_match),
            "audit_passes": (
                maximum_absolute_difference < 1e-12 and rebalance_flags_match
            ),
        }
    )

baseline_reproduction_audit = pd.DataFrame(baseline_audit_rows).set_index("portfolio")

display(baseline_reproduction_audit)

,overlapping_observations,start_date,end_date,maximum_absolute_difference,rebalance_flags_match,audit_passes
portfolio,,,,,,
Composite Score,2635,2016-01-07,2026-07-01,0.000000e+00,True,True
Fixed 50/50 Sleeves,2635,2016-01-07,2026-07-01,0.000000e+00,True,True
Pure Inverse Volatility,2635,2016-01-07,2026-07-01,1.110223e-15,True,True


## 1. Transaction-cost and rebalance-frequency sensitivity

The baseline reproduction audit passes for all three active portfolios. The robustness experiment therefore varies implementation assumptions while preserving the original signal definitions, portfolio construction rules, and drift-aware backtest engine.

The experiment uses:

- rebalance frequencies of 1, 5, 10, and 21 trading days;
- transaction costs of 0, 5, 10, 20, and 50 basis points per unit of turnover; and
- the fixed evaluation window from 7 January 2016 to 1 July 2026.

Each frequency-specific portfolio is constructed once with zero transaction costs. Alternative costs are then applied to its realised daily turnover:

$$
r^{\mathrm{net}}_t
=
r^{\mathrm{gross}}_t
-
\mathrm{turnover}_t
\frac{c}{10{,}000}.
$$

This is valid because transaction costs affect portfolio returns but do not alter the target weights, holdings drift, or subsequent turnover calculations in the backtest engine.

The original 5-day/10-bps specification remains the baseline. The grid is a sensitivity analysis rather than a search for the historically optimal implementation.

#### Construct each rebalance-frequency variant

In [5]:
robustness_reference_dates = pd.DatetimeIndex(
    exported_benchmarks.loc[
        exported_benchmarks["portfolio"] == "Composite Score",
        "date",
    ]
).sort_values()

if robustness_reference_dates.duplicated().any():
    raise ValueError("Robustness reference dates contain duplicates.")

frequency_reconstructions = {}
frequency_daily_parts = []

for rebalance_frequency in REBALANCE_FREQUENCIES:
    reconstruction = build_robustness_portfolios(
        rebalance_frequency=rebalance_frequency,
        rebalance_offset=0,
        transaction_cost_bps=0.0,
    )

    frequency_reconstructions[rebalance_frequency] = reconstruction

    for portfolio_name in ACTIVE_PORTFOLIOS:
        daily = reconstruction["daily"][portfolio_name].copy()

        daily["date"] = pd.to_datetime(daily["date"])

        daily = (
            daily.loc[daily["date"].isin(robustness_reference_dates)]
            .sort_values("date")
            .reset_index(drop=True)
        )

        portfolio_dates = pd.DatetimeIndex(daily["date"])

        if not portfolio_dates.equals(robustness_reference_dates):
            raise ValueError(
                f"{portfolio_name}, frequency "
                f"{rebalance_frequency}: dates do not "
                "match the robustness window."
            )

        if daily["date"].duplicated().any():
            raise ValueError(
                f"{portfolio_name}, frequency "
                f"{rebalance_frequency}: duplicate dates."
            )

        daily["portfolio"] = portfolio_name
        daily["rebalance_frequency"] = rebalance_frequency

        frequency_daily_parts.append(daily)

frequency_robustness_daily_gross = pd.concat(
    frequency_daily_parts,
    ignore_index=True,
)

if frequency_robustness_daily_gross.duplicated(
    [
        "portfolio",
        "rebalance_frequency",
        "date",
    ]
).any():
    raise ValueError("Duplicate portfolio-frequency-date rows found.")

frequency_construction_audit = frequency_robustness_daily_gross.groupby(
    ["portfolio", "rebalance_frequency"]
).agg(
    observations=("date", "size"),
    start_date=("date", "min"),
    end_date=("date", "max"),
    rebalance_days=("is_rebalance", "sum"),
    maximum_missing_return_weight=(
        "missing_return_weight",
        "max",
    ),
)

display(frequency_construction_audit)

observations start_date  \
portfolio               rebalance_frequency                            
Composite Score         1                            2635 2016-01-07   
                        5                            2635 2016-01-07   
                        10                           2635 2016-01-07   
                        21                           2635 2016-01-07   
Fixed 50/50 Sleeves     1                            2635 2016-01-07   
                        5                            2635 2016-01-07   
                        10                           2635 2016-01-07   
                        21                           2635 2016-01-07   
Pure Inverse Volatility 1                            2635 2016-01-07   
                        5                            2635 2016-01-07   
                        10                           2635 2016-01-07   
                        21                           2635 2016-01-07   

                                              end_date  rebalance_days  \
portfolio               rebalance_frequency                              
Composite Score         1                   2026-07-01            2635   
                        5                   2026-07-01             527   
                        10                  2026-07-01             263   
                        21                  2026-07-01             125   
Fixed 50/50 Sleeves     1                   2026-07-01            2635   
                        5                   2026-07-01             527   
                        10                  2026-07-01             263   
                        21                  2026-07-01             125   
Pure Inverse Volatility 1                   2026-07-01            2635   
                        5                   2026-07-01             527   
                        10                  2026-07-01             263   
                        21                  2026-07-01             125   

                                             maximum_missing_return_weight  
portfolio               rebalance_frequency                                 
Composite Score         1                                              0.0  
                        5                                              0.0  
                        10                                             0.0  
                        21                                             0.0  
Fixed 50/50 Sleeves     1                                              0.0  
                        5                                              0.0  
                        10                                             0.0  
                        21                                             0.0  
Pure Inverse Volatility 1                                              0.0  
                        5                                              0.0  
                        10                                             0.0  
                        21                                             0.0

#### Apply the transaction-cost grid

In [6]:
implementation_daily_parts = []

for transaction_cost_bps in TRANSACTION_COST_GRID_BPS:
    variant = frequency_robustness_daily_gross.copy()

    variant["transaction_cost_bps"] = transaction_cost_bps

    variant["transaction_cost"] = variant["turnover"] * transaction_cost_bps / 10_000.0

    variant["net_return"] = variant["gross_return"] - variant["transaction_cost"]

    implementation_daily_parts.append(variant)

implementation_robustness_daily = pd.concat(
    implementation_daily_parts,
    ignore_index=True,
)

if implementation_robustness_daily.duplicated(
    [
        "portfolio",
        "rebalance_frequency",
        "transaction_cost_bps",
        "date",
    ]
).any():
    raise ValueError("Duplicate implementation-variant dates found.")

expected_variant_count = (
    len(ACTIVE_PORTFOLIOS) * len(REBALANCE_FREQUENCIES) * len(TRANSACTION_COST_GRID_BPS)
)

actual_variant_count = (
    implementation_robustness_daily[
        [
            "portfolio",
            "rebalance_frequency",
            "transaction_cost_bps",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

if actual_variant_count != expected_variant_count:
    raise ValueError(
        f"Expected {expected_variant_count} variants, " f"found {actual_variant_count}."
    )

#### Audit the analytical cost calculation against the baseline export

In [7]:
analytical_cost_audit_rows = []

for portfolio_name in ACTIVE_PORTFOLIOS:
    reconstructed = (
        implementation_robustness_daily.loc[
            (implementation_robustness_daily["portfolio"] == portfolio_name)
            & (
                implementation_robustness_daily["rebalance_frequency"]
                == BASELINE_REBALANCE_FREQUENCY
            )
            & (
                implementation_robustness_daily["transaction_cost_bps"]
                == BASELINE_TRANSACTION_COST_BPS
            ),
            [
                "date",
                "is_rebalance",
                *BASELINE_AUDIT_COLUMNS,
            ],
        ]
        .copy()
        .sort_values("date")
    )

    exported = (
        exported_benchmarks.loc[
            exported_benchmarks["portfolio"] == portfolio_name,
            [
                "date",
                "is_rebalance",
                *BASELINE_AUDIT_COLUMNS,
            ],
        ]
        .copy()
        .sort_values("date")
    )

    aligned = reconstructed.merge(
        exported,
        on="date",
        how="inner",
        suffixes=("_reconstructed", "_exported"),
        validate="one_to_one",
    )

    maximum_absolute_difference = max(
        (aligned[f"{column}_reconstructed"] - aligned[f"{column}_exported"]).abs().max()
        for column in BASELINE_AUDIT_COLUMNS
    )

    rebalance_flags_match = (
        aligned["is_rebalance_reconstructed"].eq(aligned["is_rebalance_exported"]).all()
    )

    analytical_cost_audit_rows.append(
        {
            "portfolio": portfolio_name,
            "overlapping_observations": len(aligned),
            "maximum_absolute_difference": (maximum_absolute_difference),
            "rebalance_flags_match": (rebalance_flags_match),
            "audit_passes": (
                maximum_absolute_difference < 1e-12 and rebalance_flags_match
            ),
        }
    )

analytical_cost_reproduction_audit = pd.DataFrame(analytical_cost_audit_rows).set_index(
    "portfolio"
)

display(analytical_cost_reproduction_audit)

if not analytical_cost_reproduction_audit["audit_passes"].all():
    raise ValueError(
        "The analytical cost calculation does not " "reproduce the exported baseline."
    )

,overlapping_observations,maximum_absolute_difference,rebalance_flags_match,audit_passes
portfolio,,,,
Composite Score,2635,0.000000e+00,True,True
Fixed 50/50 Sleeves,2635,0.000000e+00,True,True
Pure Inverse Volatility,2635,1.110223e-15,True,True


#### Summarise

In [8]:
from alpha_research.backtest import summarise_backtest


implementation_summary_rows = []

summary_group_columns = [
    "portfolio",
    "rebalance_frequency",
    "transaction_cost_bps",
]

for group_values, daily in implementation_robustness_daily.groupby(
    summary_group_columns,
    sort=True,
):
    (
        portfolio_name,
        rebalance_frequency,
        transaction_cost_bps,
    ) = group_values

    gross_summary = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    implementation_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "rebalance_frequency": (rebalance_frequency),
            "transaction_cost_bps": (transaction_cost_bps),
            "observations": len(daily),
            "start_date": daily["date"].min(),
            "end_date": daily["date"].max(),
            "gross_annualised_return": (gross_summary["annualised_return"]),
            "net_annualised_return": (net_summary["annualised_return"]),
            "net_annualised_volatility": (net_summary["annualised_volatility"]),
            "net_sharpe": (net_summary["sharpe_ratio"]),
            "net_max_drawdown": (net_summary["max_drawdown"]),
            "average_daily_turnover": (net_summary["average_daily_turnover"]),
            "average_rebalance_turnover": (net_summary["average_rebalance_turnover"]),
            "total_transaction_cost": (net_summary["total_transaction_cost"]),
            "average_gross_exposure": (daily["gross_exposure"].mean()),
            "maximum_missing_return_weight": (daily["missing_return_weight"].max()),
        }
    )

implementation_robustness_summary = (
    pd.DataFrame(implementation_summary_rows)
    .sort_values(summary_group_columns)
    .reset_index(drop=True)
)

display(implementation_robustness_summary.head())

,portfolio,rebalance_frequency,transaction_cost_bps,observations,start_date,end_date,gross_annualised_return,net_annualised_return,net_annualised_volatility,net_sharpe,net_max_drawdown,average_daily_turnover,average_rebalance_turnover,total_transaction_cost,average_gross_exposure,maximum_missing_return_weight
0,Composite Score,1,0.0,2635,2016-01-07,2026-07-01,0.168953,0.168953,0.212982,0.840125,-0.269556,0.235922,0.235922,0.000000,2.0,0.0
1,Composite Score,1,5.0,2635,2016-01-07,2026-07-01,0.168953,0.134734,0.212970,0.700594,-0.292415,0.235922,0.235922,0.310828,2.0,0.0
2,Composite Score,1,10.0,2635,2016-01-07,2026-07-01,0.168953,0.101512,0.212963,0.561033,-0.329433,0.235922,0.235922,0.621656,2.0,0.0
3,Composite Score,1,20.0,2635,2016-01-07,2026-07-01,0.168953,0.037943,0.212963,0.281865,-0.471828,0.235922,0.235922,1.243311,2.0,0.0
4,Composite Score,1,50.0,2635,2016-01-07,2026-07-01,0.168953,-0.131682,0.213079,-0.555337,-0.855122,0.235922,0.235922,3.108278,2.0,0.0


#### Prepare SPY as a frequency-independent reference

In [9]:
spy_daily = (
    exported_benchmarks.loc[exported_benchmarks["portfolio"] == "SPY Buy and Hold"]
    .copy()
    .sort_values("date")
    .reset_index(drop=True)
)

spy_dates = pd.DatetimeIndex(spy_daily["date"])

if not spy_dates.equals(robustness_reference_dates):
    raise ValueError("SPY dates do not match the robustness window.")

spy_cost_rows = []

for transaction_cost_bps in TRANSACTION_COST_GRID_BPS:
    variant = spy_daily.copy()

    variant["transaction_cost"] = variant["turnover"] * transaction_cost_bps / 10_000.0

    variant["net_return"] = variant["gross_return"] - variant["transaction_cost"]

    net_summary = summarise_backtest(
        variant,
        return_column="net_return",
    ).iloc[0]

    spy_cost_rows.append(
        {
            "transaction_cost_bps": (transaction_cost_bps),
            "net_annualised_return": (net_summary["annualised_return"]),
            "net_annualised_volatility": (net_summary["annualised_volatility"]),
            "net_sharpe": (net_summary["sharpe_ratio"]),
            "net_max_drawdown": (net_summary["max_drawdown"]),
            "average_daily_turnover": (net_summary["average_daily_turnover"]),
            "total_transaction_cost": (net_summary["total_transaction_cost"]),
        }
    )

spy_cost_sensitivity = pd.DataFrame(spy_cost_rows)

#### Display results

In [10]:
baseline_cost_frequency_summary = implementation_robustness_summary.loc[
    implementation_robustness_summary["transaction_cost_bps"]
    == BASELINE_TRANSACTION_COST_BPS
].set_index(["portfolio", "rebalance_frequency"])[
    [
        "gross_annualised_return",
        "net_annualised_return",
        "net_annualised_volatility",
        "net_sharpe",
        "net_max_drawdown",
        "average_daily_turnover",
        "average_rebalance_turnover",
        "total_transaction_cost",
        "average_gross_exposure",
    ]
]

annualised_return_sensitivity = implementation_robustness_summary.pivot(
    index=[
        "portfolio",
        "rebalance_frequency",
    ],
    columns="transaction_cost_bps",
    values="net_annualised_return",
)

sharpe_sensitivity = implementation_robustness_summary.pivot(
    index=[
        "portfolio",
        "rebalance_frequency",
    ],
    columns="transaction_cost_bps",
    values="net_sharpe",
)

display(baseline_cost_frequency_summary.round(4))

display(annualised_return_sensitivity.round(4))

display(sharpe_sensitivity.round(4))

display(spy_cost_sensitivity.set_index("transaction_cost_bps").round(4))

gross_annualised_return  \
portfolio               rebalance_frequency                            
Composite Score         1                                     0.1690   
                        5                                     0.1678   
                        10                                    0.1692   
                        21                                    0.1763   
Fixed 50/50 Sleeves     1                                     0.1337   
                        5                                     0.1280   
                        10                                    0.1260   
                        21                                    0.1191   
Pure Inverse Volatility 1                                     0.1359   
                        5                                     0.1301   
                        10                                    0.1282   
                        21                                    0.1326   

                                             net_annualised_return  \
portfolio               rebalance_frequency                          
Composite Score         1                                   0.1015   
                        5                                   0.1374   
                        10                                  0.1479   
                        21                                  0.1613   
Fixed 50/50 Sleeves     1                                   0.0802   
                        5                                   0.1035   
                        10                                  0.1084   
                        21                                  0.1073   
Pure Inverse Volatility 1                                   0.0796   
                        5                                   0.1041   
                        10                                  0.1096   
                        21                                  0.1199   

                                             net_annualised_volatility  \
portfolio               rebalance_frequency                              
Composite Score         1                                       0.2130   
                        5                                       0.2133   
                        10                                      0.2120   
                        21                                      0.2117   
Fixed 50/50 Sleeves     1                                       0.1683   
                        5                                       0.1684   
                        10                                      0.1677   
                        21                                      0.1662   
Pure Inverse Volatility 1                                       0.1623   
                        5                                       0.1629   
                        10                                      0.1625   
                        21                                      0.1618   

                                             net_sharpe  net_max_drawdown  \
portfolio               rebalance_frequency                                 
Composite Score         1                        0.5610           -0.3294   
                        5                        0.7106           -0.3063   
                        10                       0.7570           -0.3171   
                        21                       0.8128           -0.2924   
Fixed 50/50 Sleeves     1                        0.5428           -0.2585   
                        5                        0.6695           -0.2566   
                        10                       0.6983           -0.2568   
                        21                       0.6970           -0.2407   
Pure Inverse Volatility 1                        0.5531           -0.2443   
                        5                        0.6899           -0.1957   
                        10                       0.7215           -0.2177   
                 

transaction_cost_bps                           0.0     5.0     10.0    20.0  \
portfolio               rebalance_frequency                                   
Composite Score         1                    0.1690  0.1347  0.1015  0.0379   
                        5                    0.1678  0.1525  0.1374  0.1078   
                        10                   0.1692  0.1585  0.1479  0.1269   
                        21                   0.1763  0.1687  0.1613  0.1464   
Fixed 50/50 Sleeves     1                    0.1337  0.1066  0.0802  0.0292   
                        5                    0.1280  0.1157  0.1035  0.0796   
                        10                   0.1260  0.1172  0.1084  0.0912   
                        21                   0.1191  0.1132  0.1073  0.0957   
Pure Inverse Volatility 1                    0.1359  0.1074  0.0796  0.0260   
                        5                    0.1301  0.1170  0.1041  0.0787   
                        10                   0.1282  0.1189  0.1096  0.0913   
                        21                   0.1326  0.1263  0.1199  0.1074   

transaction_cost_bps                           50.0  
portfolio               rebalance_frequency          
Composite Score         1                   -0.1317  
                        5                    0.0234  
                        10                   0.0662  
                        21                   0.1030  
Fixed 50/50 Sleeves     1                   -0.1098  
                        5                    0.0107  
                        10                   0.0408  
                        21                   0.0614  
Pure Inverse Volatility 1                   -0.1194  
                        5                    0.0058  
                        10                   0.0380  
                        21                   0.0704

transaction_cost_bps                           0.0     5.0     10.0    20.0  \
portfolio               rebalance_frequency                                   
Composite Score         1                    0.8401  0.7006  0.5610  0.2819   
                        5                    0.8343  0.7725  0.7106  0.5867   
                        10                   0.8439  0.8005  0.7570  0.6698   
                        21                   0.8736  0.8432  0.8128  0.7517   
Fixed 50/50 Sleeves     1                    0.8300  0.6864  0.5428  0.2555   
                        5                    0.8000  0.7348  0.6695  0.5389   
                        10                   0.7920  0.7452  0.6983  0.6044   
                        21                   0.7609  0.7290  0.6970  0.6330   
Pure Inverse Volatility 1                    0.8667  0.7099  0.5531  0.2394   
                        5                    0.8330  0.7615  0.6899  0.5466   
                        10                   0.8239  0.7727  0.7215  0.6189   
                        21                   0.8514  0.8165  0.7815  0.7113   

transaction_cost_bps                           50.0  
portfolio               rebalance_frequency          
Composite Score         1                   -0.5553  
                        5                    0.2154  
                        10                   0.4079  
                        21                   0.5679  
Fixed 50/50 Sleeves     1                   -0.6065  
                        5                    0.1477  
                        10                   0.3224  
                        21                   0.4407  
Pure Inverse Volatility 1                   -0.7016  
                        5                    0.1176  
                        10                   0.3107  
                        21                   0.5002

,net_annualised_return,net_annualised_volatility,net_sharpe,net_max_drawdown,average_daily_turnover,total_transaction_cost
transaction_cost_bps,,,,,,
0.0,0.1555,0.1783,0.9004,-0.3372,0.0004,0.0000
5.0,0.1555,0.1783,0.9001,-0.3372,0.0004,0.0005
10.0,0.1554,0.1783,0.8998,-0.3372,0.0004,0.0010
20.0,0.1553,0.1783,0.8992,-0.3372,0.0004,0.0020
50.0,0.1550,0.1783,0.8975,-0.3372,0.0004,0.0050


### Initial findings

The sensitivity grid reveals a strong and consistent trade-off between signal freshness and implementation cost.

#### Rebalance frequency

Daily rebalancing generates the highest turnover and performs poorly after transaction costs:

- at 10 bps, annual transaction-cost drag is approximately 5–7 percentage points across the active portfolios;
- at 50 bps, every daily-rebalanced portfolio has a negative annualised return; and
- the increase in trading frequency does not produce a corresponding improvement in gross performance.

Extending the holding period substantially reduces average daily turnover. Average turnover per rebalance increases because more holdings change between decisions, but the lower number of rebalance events more than offsets this effect.

At the baseline 10-bps cost:

- **Composite Score** improves from a 0.561 Sharpe ratio with daily rebalancing to 0.757 at 10 days and 0.813 at 21 days.
- **Fixed 50/50 Sleeves** reaches similar results at 10 and 21 days, with Sharpe ratios of 0.698 and 0.697 respectively.
- **Pure Inverse Volatility** improves from 0.553 daily to 0.722 at 10 days and 0.782 at 21 days.

These improvements are not caused solely by lower transaction costs. Gross Composite Score performance remains stable across frequencies and is strongest at 21 days. Pure Inverse Volatility also retains similar gross returns at longer frequencies. Fixed 50/50 experiences some gross-performance decay, although the cost savings compensate for most of it.

#### Transaction-cost sensitivity

The slower implementations remain profitable under every tested cost assumption.

At 50 bps:

- 21-day Composite Score earns 10.30% annually with a 0.568 Sharpe ratio;
- 21-day Fixed 50/50 earns 6.14% with a 0.441 Sharpe ratio; and
- 21-day Pure Inverse Volatility earns 7.04% with a 0.500 Sharpe ratio.

The 5-day implementations remain profitable at 50 bps but retain little economic advantage. The 10- and 21-day variants are materially more resilient.

#### Comparison with SPY

SPY is almost insensitive to the transaction-cost assumptions because it is implemented as buy-and-hold. At the baseline 10-bps cost, it earns 15.54% annually with a 0.900 Sharpe ratio.

The 21-day Composite Score slightly exceeds SPY's annualised return, earning 16.13%, but has a lower Sharpe ratio of 0.813. The active portfolios nevertheless provide shallower maximum drawdowns:

- 21-day Composite Score: −29.24%;
- 21-day Fixed 50/50: −24.07%;
- 21-day Pure Inverse Volatility: −22.31%;
- SPY Buy and Hold: −33.72%.

These are not leverage-matched comparisons. Average gross exposure is approximately 2.0 for Composite Score, 1.54 for Fixed 50/50, and 1.61 for Pure Inverse Volatility, compared with approximately 1.0 for SPY. Later analysis should therefore compare volatility-scaled performance and capital efficiency rather than annualised return alone.

#### Interim interpretation

The evidence rejects daily rebalancing as a plausible implementation. Ten- and 21-day schedules provide a much better turnover–performance balance.

However, the apparent strength of the 21-day variants may depend on the particular rebalance phase used by offset 0. All possible rebalance offsets must be tested before determining whether the monthly-frequency result is representative.

### Rebalance offset sensitivity

In [11]:
OFFSET_SENSITIVITY_COST_BPS = BASELINE_TRANSACTION_COST_BPS

offset_daily_parts = []
security_trade_detail_parts = []

for rebalance_frequency in REBALANCE_FREQUENCIES:
    for rebalance_offset in range(rebalance_frequency):
        if rebalance_offset == 0:
            reconstruction = frequency_reconstructions[rebalance_frequency]
        else:
            reconstruction = build_robustness_portfolios(
                rebalance_frequency=(rebalance_frequency),
                rebalance_offset=rebalance_offset,
                transaction_cost_bps=0.0,
            )

        for portfolio_name in ACTIVE_PORTFOLIOS:
            daily = reconstruction["daily"][portfolio_name].copy()

            trade_detail = (
                reconstruction["trades"][portfolio_name]
                .loc[lambda df: df["date"].isin(robustness_reference_dates)]
                .rename(columns={"trade": "trade_weight"})
                .assign(
                    portfolio=portfolio_name,
                    rebalance_frequency=rebalance_frequency,
                    rebalance_offset=rebalance_offset,
                )
            )

            security_trade_detail_parts.append(trade_detail)

            daily["date"] = pd.to_datetime(daily["date"])

            daily = (
                daily.loc[daily["date"].isin(robustness_reference_dates)]
                .sort_values("date")
                .reset_index(drop=True)
            )

            portfolio_dates = pd.DatetimeIndex(daily["date"])

            if not portfolio_dates.equals(robustness_reference_dates):
                raise ValueError(
                    f"{portfolio_name}, frequency "
                    f"{rebalance_frequency}, offset "
                    f"{rebalance_offset}: dates do not "
                    "match the robustness window."
                )

            daily["transaction_cost"] = (
                daily["turnover"] * OFFSET_SENSITIVITY_COST_BPS / 10_000.0
            )

            daily["net_return"] = daily["gross_return"] - daily["transaction_cost"]

            daily["gross_cumulative_return"] = (1.0 + daily["gross_return"]).cumprod()

            daily["net_cumulative_return"] = (1.0 + daily["net_return"]).cumprod()

            daily["portfolio"] = portfolio_name
            daily["rebalance_frequency"] = rebalance_frequency
            daily["rebalance_offset"] = rebalance_offset
            daily["transaction_cost_bps"] = OFFSET_SENSITIVITY_COST_BPS

            offset_daily_parts.append(daily)

rebalance_offset_daily = pd.concat(
    offset_daily_parts,
    ignore_index=True,
)

offset_key_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
    "date",
]

if rebalance_offset_daily.duplicated(offset_key_columns).any():
    raise ValueError("Duplicate portfolio-frequency-offset-date " "rows found.")

expected_offset_variants = len(ACTIVE_PORTFOLIOS) * sum(REBALANCE_FREQUENCIES)

actual_offset_variants = (
    rebalance_offset_daily[
        [
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

if actual_offset_variants != expected_offset_variants:
    raise ValueError(
        f"Expected {expected_offset_variants} "
        f"offset variants, found "
        f"{actual_offset_variants}."
    )

security_trade_detail = (
    pd.concat(
        security_trade_detail_parts,
        ignore_index=True,
    )[
        [
            "date",
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
            "ticker",
            "trade_weight",
        ]
    ]
    .sort_values(
        [
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
            "date",
            "ticker",
        ]
    )
    .reset_index(drop=True)
)

security_trade_key_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
    "date",
    "ticker",
]

if security_trade_detail.duplicated(security_trade_key_columns).any():
    raise ValueError("Duplicate security-level trade rows found.")

#### Audit rebalance offset construction

In [12]:
rebalance_offset_construction_audit = rebalance_offset_daily.groupby(
    [
        "portfolio",
        "rebalance_frequency",
        "rebalance_offset",
    ]
).agg(
    observations=("date", "size"),
    start_date=("date", "min"),
    end_date=("date", "max"),
    rebalance_days=("is_rebalance", "sum"),
    maximum_missing_return_weight=(
        "missing_return_weight",
        "max",
    ),
)

expected_observations = len(robustness_reference_dates)

if not (
    rebalance_offset_construction_audit["observations"] == expected_observations
).all():
    raise ValueError(
        "At least one offset variant has an " "unexpected number of observations."
    )

display(
    rebalance_offset_construction_audit.groupby(
        ["portfolio", "rebalance_frequency"]
    ).agg(
        offsets=("observations", "size"),
        minimum_rebalance_days=(
            "rebalance_days",
            "min",
        ),
        maximum_rebalance_days=(
            "rebalance_days",
            "max",
        ),
        maximum_missing_return_weight=(
            "maximum_missing_return_weight",
            "max",
        ),
    )
)

offsets  minimum_rebalance_days  \
portfolio               rebalance_frequency                                    
Composite Score         1                          1                    2635   
                        5                          5                     527   
                        10                        10                     263   
                        21                        21                     125   
Fixed 50/50 Sleeves     1                          1                    2635   
                        5                          5                     527   
                        10                        10                     263   
                        21                        21                     125   
Pure Inverse Volatility 1                          1                    2635   
                        5                          5                     527   
                        10                        10                     263   
                        21                        21                     125   

                                             maximum_rebalance_days  \
portfolio               rebalance_frequency                           
Composite Score         1                                      2635   
                        5                                       527   
                        10                                      264   
                        21                                      126   
Fixed 50/50 Sleeves     1                                      2635   
                        5                                       527   
                        10                                      264   
                        21                                      126   
Pure Inverse Volatility 1                                      2635   
                        5                                       527   
                        10                                      264   
                        21                                      126   

                                             maximum_missing_return_weight  
portfolio               rebalance_frequency                                 
Composite Score         1                                              0.0  
                        5                                              0.0  
                        10                                             0.0  
                        21                                             0.0  
Fixed 50/50 Sleeves     1                                              0.0  
                        5                                              0.0  
                        10                                             0.0  
                        21                                             0.0  
Pure Inverse Volatility 1                                              0.0  
                        5                                              0.0  
                        10                                             0.0  
                        21                                             0.0

#### Summarise each offset

In [13]:
rebalance_offset_summary_rows = []

offset_group_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
]

for group_values, daily in rebalance_offset_daily.groupby(
    offset_group_columns,
    sort=True,
):
    (
        portfolio_name,
        rebalance_frequency,
        rebalance_offset,
    ) = group_values

    gross_summary = summarise_backtest(
        daily,
        return_column="gross_return",
    ).iloc[0]

    net_summary = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    rebalance_offset_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "rebalance_frequency": (rebalance_frequency),
            "rebalance_offset": rebalance_offset,
            "gross_annualised_return": (gross_summary["annualised_return"]),
            "net_annualised_return": (net_summary["annualised_return"]),
            "net_annualised_volatility": (net_summary["annualised_volatility"]),
            "net_sharpe": (net_summary["sharpe_ratio"]),
            "net_max_drawdown": (net_summary["max_drawdown"]),
            "average_daily_turnover": (net_summary["average_daily_turnover"]),
            "average_rebalance_turnover": (net_summary["average_rebalance_turnover"]),
            "total_transaction_cost": (net_summary["total_transaction_cost"]),
            "average_gross_exposure": (daily["gross_exposure"].mean()),
        }
    )

rebalance_offset_summary = (
    pd.DataFrame(rebalance_offset_summary_rows)
    .sort_values(offset_group_columns)
    .reset_index(drop=True)
)

In [14]:
OFFSET_AUDIT_COLUMNS = [
    "gross_annualised_return",
    "net_annualised_return",
    "net_annualised_volatility",
    "net_sharpe",
    "net_max_drawdown",
    "average_daily_turnover",
    "average_rebalance_turnover",
    "total_transaction_cost",
    "average_gross_exposure",
]

offset_zero = rebalance_offset_summary.loc[
    rebalance_offset_summary["rebalance_offset"] == 0,
    [
        "portfolio",
        "rebalance_frequency",
        *OFFSET_AUDIT_COLUMNS,
    ],
]

previous_grid_baseline = implementation_robustness_summary.loc[
    implementation_robustness_summary["transaction_cost_bps"]
    == OFFSET_SENSITIVITY_COST_BPS,
    [
        "portfolio",
        "rebalance_frequency",
        *OFFSET_AUDIT_COLUMNS,
    ],
]

offset_zero_audit = offset_zero.merge(
    previous_grid_baseline,
    on=[
        "portfolio",
        "rebalance_frequency",
    ],
    suffixes=("_offset", "_previous"),
    validate="one_to_one",
)

offset_zero_maximum_difference = max(
    (offset_zero_audit[f"{column}_offset"] - offset_zero_audit[f"{column}_previous"])
    .abs()
    .max()
    for column in OFFSET_AUDIT_COLUMNS
)

if offset_zero_maximum_difference >= 1e-12:
    raise ValueError(
        "Offset-zero results do not reproduce " "the previous frequency grid."
    )

print(
    "Maximum offset-zero audit difference:",
    offset_zero_maximum_difference,
)

Maximum offset-zero audit difference: 0.0


#### Measure the range across offsets

In [15]:
rebalance_offset_range_summary = rebalance_offset_summary.groupby(
    ["portfolio", "rebalance_frequency"]
).agg(
    offset_count=(
        "rebalance_offset",
        "nunique",
    ),
    mean_net_annualised_return=(
        "net_annualised_return",
        "mean",
    ),
    median_net_annualised_return=(
        "net_annualised_return",
        "median",
    ),
    minimum_net_annualised_return=(
        "net_annualised_return",
        "min",
    ),
    maximum_net_annualised_return=(
        "net_annualised_return",
        "max",
    ),
    standard_deviation_of_return=(
        "net_annualised_return",
        "std",
    ),
    mean_net_sharpe=(
        "net_sharpe",
        "mean",
    ),
    median_net_sharpe=(
        "net_sharpe",
        "median",
    ),
    minimum_net_sharpe=(
        "net_sharpe",
        "min",
    ),
    maximum_net_sharpe=(
        "net_sharpe",
        "max",
    ),
    standard_deviation_of_sharpe=(
        "net_sharpe",
        "std",
    ),
    worst_max_drawdown=(
        "net_max_drawdown",
        "min",
    ),
    best_max_drawdown=(
        "net_max_drawdown",
        "max",
    ),
    mean_daily_turnover=(
        "average_daily_turnover",
        "mean",
    ),
)

In [16]:
offset_zero_comparison = (
    rebalance_offset_summary.loc[
        rebalance_offset_summary["rebalance_offset"] == 0,
        [
            "portfolio",
            "rebalance_frequency",
            "net_annualised_return",
            "net_sharpe",
            "net_max_drawdown",
        ],
    ]
    .rename(
        columns={
            "net_annualised_return": ("offset_0_net_annualised_return"),
            "net_sharpe": "offset_0_net_sharpe",
            "net_max_drawdown": ("offset_0_net_max_drawdown"),
        }
    )
    .set_index(["portfolio", "rebalance_frequency"])
    .join(
        rebalance_offset_range_summary[
            [
                "median_net_annualised_return",
                "median_net_sharpe",
            ]
        ]
    )
)

offset_zero_comparison["return_difference_from_median"] = (
    offset_zero_comparison["offset_0_net_annualised_return"]
    - offset_zero_comparison["median_net_annualised_return"]
)

offset_zero_comparison["sharpe_difference_from_median"] = (
    offset_zero_comparison["offset_0_net_sharpe"]
    - offset_zero_comparison["median_net_sharpe"]
)

#### Display the results

In [17]:
display(rebalance_offset_range_summary.round(4))

display(offset_zero_comparison.round(4))

monthly_offset_detail = rebalance_offset_summary.loc[
    rebalance_offset_summary["rebalance_frequency"] == 21
].pivot(
    index="portfolio",
    columns="rebalance_offset",
    values="net_sharpe",
)

display(monthly_offset_detail.round(4))

offset_count  \
portfolio               rebalance_frequency                 
Composite Score         1                               1   
                        5                               5   
                        10                             10   
                        21                             21   
Fixed 50/50 Sleeves     1                               1   
                        5                               5   
                        10                             10   
                        21                             21   
Pure Inverse Volatility 1                               1   
                        5                               5   
                        10                             10   
                        21                             21   

                                             mean_net_annualised_return  \
portfolio               rebalance_frequency                               
Composite Score         1                                        0.1015   
                        5                                        0.1303   
                        10                                       0.1483   
                        21                                       0.1584   
Fixed 50/50 Sleeves     1                                        0.0802   
                        5                                        0.1020   
                        10                                       0.1050   
                        21                                       0.1016   
Pure Inverse Volatility 1                                        0.0796   
                        5                                        0.1048   
                        10                                       0.1110   
                        21                                       0.1076   

                                             median_net_annualised_return  \
portfolio               rebalance_frequency                                 
Composite Score         1                                          0.1015   
                        5                                          0.1326   
                        10                                         0.1470   
                        21                                         0.1577   
Fixed 50/50 Sleeves     1                                          0.0802   
                        5                                          0.1024   
                        10                                         0.1047   
                        21                                         0.1007   
Pure Inverse Volatility 1                                          0.0796   
                        5                                          0.1041   
                        10                                         0.1113   
                        21                                         0.1094   

                                             minimum_net_annualised_return  \
portfolio               rebalance_frequency                                  
Composite Score         1                                           0.1015   
                        5                                           0.1148   
                        10                                          0.1337   
                        21                                          0.1346   
Fixed 50/50 Sleeves     1                                           0.0802   
                        5                                           0.0995   
                        10                                          0.0944   
                        21                                          0.0928   
Pure Inverse Volatility 1                                           0.0796   
                        5                                           0.1030   
                        10                                          0.0983   
                        21                    

offset_0_net_annualised_return  \
portfolio               rebalance_frequency                                   
Composite Score         1                                            0.1015   
                        5                                            0.1374   
                        10                                           0.1479   
                        21                                           0.1613   
Fixed 50/50 Sleeves     1                                            0.0802   
                        5                                            0.1035   
                        10                                           0.1084   
                        21                                           0.1073   
Pure Inverse Volatility 1                                            0.0796   
                        5                                            0.1041   
                        10                                           0.1096   
                        21                                           0.1199   

                                             offset_0_net_sharpe  \
portfolio               rebalance_frequency                        
Composite Score         1                                 0.5610   
                        5                                 0.7106   
                        10                                0.7570   
                        21                                0.8128   
Fixed 50/50 Sleeves     1                                 0.5428   
                        5                                 0.6695   
                        10                                0.6983   
                        21                                0.6970   
Pure Inverse Volatility 1                                 0.5531   
                        5                                 0.6899   
                        10                                0.7215   
                        21                                0.7815   

                                             offset_0_net_max_drawdown  \
portfolio               rebalance_frequency                              
Composite Score         1                                      -0.3294   
                        5                                      -0.3063   
                        10                                     -0.3171   
                        21                                     -0.2924   
Fixed 50/50 Sleeves     1                                      -0.2585   
                        5                                      -0.2566   
                        10                                     -0.2568   
                        21                                     -0.2407   
Pure Inverse Volatility 1                                      -0.2443   
                        5                                      -0.1957   
                        10                                     -0.2177   
                        21                                     -0.2231   

                                             median_net_annualised_return  \
portfolio               rebalance_frequency                                 
Composite Score         1                                          0.1015   
                        5                                          0.1326   
                        10                                         0.1470   
                        21                                         0.1577   
Fixed 50/50 Sleeves     1                                          0.0802   
                        5                                          0.1024   
                        10                                         0.1047   
                        21                                         0.1007   
Pure Inverse Volatility 1                                          0.0796   
                        5                                          0.1041   
                        

rebalance_offset,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
portfolio,,,,,,,,,,,,,,,,,,,,,
Composite Score,0.8128,0.8327,0.8135,0.8135,0.8272,0.7905,0.8010,0.7715,0.8777,0.8593,...,0.8198,0.7857,0.7625,0.7686,0.7552,0.7261,0.7013,0.7999,0.8502,0.8818
Fixed 50/50 Sleeves,0.6970,0.6589,0.6934,0.6400,0.6691,0.6299,0.6615,0.6449,0.6315,0.6249,...,0.7128,0.7210,0.6677,0.6553,0.6150,0.6489,0.6276,0.6541,0.6820,0.6804
Pure Inverse Volatility,0.7815,0.7187,0.7585,0.6970,0.7140,0.6440,0.6555,0.6471,0.6456,0.6481,...,0.7571,0.7675,0.7205,0.7056,0.6779,0.7216,0.6927,0.7257,0.7444,0.7427


### Rebalance-offset findings

The offset analysis confirms that the improvement from slower rebalancing is not merely an artefact of choosing offset 0. However, the preferred frequency differs across portfolio constructions.

#### Composite Score

Composite Score improves consistently as the holding period increases:

- mean net Sharpe rises from 0.682 at 5 days to 0.759 at 10 days and 0.802 at 21 days;
- the median 21-day Sharpe is 0.801;
- every 21-day offset remains profitable, with Sharpe ratios between 0.701 and 0.882; and
- offset 0 is close to the median, exceeding it by only 0.012 Sharpe.

The worst 21-day Sharpe is effectively equal to the worst 10-day Sharpe, while its mean and median performance are substantially higher. The monthly Composite result is therefore representative of a robust frequency effect rather than a favourable starting phase.

#### Fixed 50/50 Sleeves

Fixed 50/50 performs best around the 10-day frequency:

- mean Sharpe is 0.661 at 5 days, 0.680 at 10 days, and 0.663 at 21 days;
- mean annualised return is 10.20%, 10.50%, and 10.16% respectively; and
- the worst 10-day Sharpe of 0.624 is slightly stronger than the worst 21-day Sharpe of 0.615.

Offset 0 makes the 21-day implementation appear somewhat better than its typical phase. Its Sharpe of 0.697 exceeds the monthly median of 0.659 by 0.038. There is therefore no robust evidence that extending this portfolio from 10 to 21 days improves performance.

#### Pure Inverse Volatility

Pure Inverse Volatility also favours the 10-day region:

- mean Sharpe is 0.694 at 5 days, 0.729 at 10 days, and 0.709 at 21 days;
- median Sharpe is 0.690, 0.732, and 0.719 respectively; and
- the worst 10-day Sharpe of 0.661 exceeds the worst 21-day Sharpe of 0.644.

The offset-0 monthly result is particularly optimistic. Its 0.782 Sharpe is the maximum among the 21 monthly phases and exceeds the median by 0.063. Consequently, the previously observed strength of the offset-0 21-day Pure Inverse Volatility portfolio should not be treated as representative.

#### Overall interpretation

Daily rebalancing remains clearly unattractive after costs. The robust implementation region is approximately 10–21 trading days, but the evidence does not support imposing one frequency on every portfolio:

- 21 days is a credible implementation for Composite Score;
- 10 days provides the strongest phase-robust trade-off for Fixed 50/50 and Pure Inverse Volatility; and
- the original 5-day specification remains the pre-specified research baseline rather than being retrospectively replaced by the best historical variant.

Rebalance offsets are highly overlapping implementations evaluated over the same sample. Their dispersion measures sensitivity to calendar phase; it is not a sampling-error estimate or a statistical confidence interval.

### Combine transaction-cost and offset sensitivity

In [18]:
offset_cost_daily_parts = []

for transaction_cost_bps in TRANSACTION_COST_GRID_BPS:
    variant = rebalance_offset_daily.copy()

    variant["transaction_cost_bps"] = transaction_cost_bps

    variant["transaction_cost"] = variant["turnover"] * transaction_cost_bps / 10_000.0

    variant["net_return"] = variant["gross_return"] - variant["transaction_cost"]

    offset_cost_daily_parts.append(variant)

offset_cost_daily = pd.concat(
    offset_cost_daily_parts,
    ignore_index=True,
)

offset_cost_key_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
    "transaction_cost_bps",
    "date",
]

if offset_cost_daily.duplicated(offset_cost_key_columns).any():
    raise ValueError("Duplicate offset-cost-date rows found.")

expected_offset_cost_variants = (
    len(ACTIVE_PORTFOLIOS) * sum(REBALANCE_FREQUENCIES) * len(TRANSACTION_COST_GRID_BPS)
)

actual_offset_cost_variants = (
    offset_cost_daily[
        [
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
            "transaction_cost_bps",
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

if actual_offset_cost_variants != expected_offset_cost_variants:
    raise ValueError(
        f"Expected {expected_offset_cost_variants} "
        f"offset-cost variants, found "
        f"{actual_offset_cost_variants}."
    )

In [19]:
offset_cost_summary_rows = []

offset_cost_group_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
    "transaction_cost_bps",
]

for group_values, daily in offset_cost_daily.groupby(
    offset_cost_group_columns,
    sort=True,
):
    (
        portfolio_name,
        rebalance_frequency,
        rebalance_offset,
        transaction_cost_bps,
    ) = group_values

    net_summary = summarise_backtest(
        daily,
        return_column="net_return",
    ).iloc[0]

    offset_cost_summary_rows.append(
        {
            "portfolio": portfolio_name,
            "rebalance_frequency": (rebalance_frequency),
            "rebalance_offset": rebalance_offset,
            "transaction_cost_bps": (transaction_cost_bps),
            "net_annualised_return": (net_summary["annualised_return"]),
            "net_annualised_volatility": (net_summary["annualised_volatility"]),
            "net_sharpe": (net_summary["sharpe_ratio"]),
            "net_max_drawdown": (net_summary["max_drawdown"]),
            "average_daily_turnover": (net_summary["average_daily_turnover"]),
            "total_transaction_cost": (net_summary["total_transaction_cost"]),
        }
    )

offset_cost_summary = (
    pd.DataFrame(offset_cost_summary_rows)
    .sort_values(offset_cost_group_columns)
    .reset_index(drop=True)
)

In [20]:
offset_cost_range_summary = offset_cost_summary.groupby(
    [
        "portfolio",
        "rebalance_frequency",
        "transaction_cost_bps",
    ]
).agg(
    offset_count=(
        "rebalance_offset",
        "nunique",
    ),
    mean_net_annualised_return=(
        "net_annualised_return",
        "mean",
    ),
    median_net_annualised_return=(
        "net_annualised_return",
        "median",
    ),
    minimum_net_annualised_return=(
        "net_annualised_return",
        "min",
    ),
    maximum_net_annualised_return=(
        "net_annualised_return",
        "max",
    ),
    mean_net_sharpe=(
        "net_sharpe",
        "mean",
    ),
    median_net_sharpe=(
        "net_sharpe",
        "median",
    ),
    minimum_net_sharpe=(
        "net_sharpe",
        "min",
    ),
    maximum_net_sharpe=(
        "net_sharpe",
        "max",
    ),
    worst_max_drawdown=(
        "net_max_drawdown",
        "min",
    ),
    mean_daily_turnover=(
        "average_daily_turnover",
        "mean",
    ),
)

In [21]:
offset_cost_baseline = offset_cost_range_summary.xs(
    OFFSET_SENSITIVITY_COST_BPS,
    level="transaction_cost_bps",
)

OFFSET_COST_AUDIT_COLUMNS = [
    "mean_net_annualised_return",
    "median_net_annualised_return",
    "minimum_net_annualised_return",
    "maximum_net_annualised_return",
    "mean_net_sharpe",
    "median_net_sharpe",
    "minimum_net_sharpe",
    "maximum_net_sharpe",
    "worst_max_drawdown",
    "mean_daily_turnover",
]

offset_cost_audit_difference = max(
    (offset_cost_baseline[column] - rebalance_offset_range_summary[column]).abs().max()
    for column in OFFSET_COST_AUDIT_COLUMNS
)

if offset_cost_audit_difference >= 1e-12:
    raise ValueError(
        "The 10-bps offset-cost results do not "
        "reproduce the previous offset summary."
    )

print(
    "Maximum offset-cost audit difference:",
    offset_cost_audit_difference,
)

Maximum offset-cost audit difference: 0.0


In [22]:
phase_averaged_return_sensitivity = offset_cost_range_summary[
    "mean_net_annualised_return"
].unstack("transaction_cost_bps")

phase_averaged_sharpe_sensitivity = offset_cost_range_summary[
    "mean_net_sharpe"
].unstack("transaction_cost_bps")

worst_offset_sharpe_sensitivity = offset_cost_range_summary[
    "minimum_net_sharpe"
].unstack("transaction_cost_bps")

display(phase_averaged_return_sensitivity.round(4))

display(phase_averaged_sharpe_sensitivity.round(4))

display(worst_offset_sharpe_sensitivity.round(4))

transaction_cost_bps                           0.0     5.0     10.0    20.0  \
portfolio               rebalance_frequency                                   
Composite Score         1                    0.1690  0.1347  0.1015  0.0379   
                        5                    0.1607  0.1454  0.1303  0.1006   
                        10                   0.1702  0.1592  0.1483  0.1269   
                        21                   0.1734  0.1658  0.1584  0.1435   
Fixed 50/50 Sleeves     1                    0.1337  0.1066  0.0802  0.0292   
                        5                    0.1266  0.1142  0.1020  0.0780   
                        10                   0.1223  0.1136  0.1050  0.0880   
                        21                   0.1132  0.1074  0.1016  0.0900   
Pure Inverse Volatility 1                    0.1359  0.1074  0.0796  0.0260   
                        5                    0.1308  0.1177  0.1048  0.0792   
                        10                   0.1295  0.1202  0.1110  0.0928   
                        21                   0.1201  0.1139  0.1076  0.0952   

transaction_cost_bps                           50.0  
portfolio               rebalance_frequency          
Composite Score         1                   -0.1317  
                        5                    0.0161  
                        10                   0.0648  
                        21                   0.1000  
Fixed 50/50 Sleeves     1                   -0.1098  
                        5                    0.0089  
                        10                   0.0383  
                        21                   0.0561  
Pure Inverse Volatility 1                   -0.1194  
                        5                    0.0061  
                        10                   0.0399  
                        21                   0.0588

transaction_cost_bps                           0.0     5.0     10.0    20.0  \
portfolio               rebalance_frequency                                   
Composite Score         1                    0.8401  0.7006  0.5610  0.2819   
                        5                    0.8069  0.7444  0.6819  0.5568   
                        10                   0.8476  0.8031  0.7586  0.6694   
                        21                   0.8634  0.8328  0.8022  0.7409   
Fixed 50/50 Sleeves     1                    0.8300  0.6864  0.5428  0.2555   
                        5                    0.7924  0.7269  0.6614  0.5302   
                        10                   0.7727  0.7263  0.6798  0.5868   
                        21                   0.7255  0.6940  0.6625  0.5993   
Pure Inverse Volatility 1                    0.8667  0.7099  0.5531  0.2394   
                        5                    0.8374  0.7656  0.6938  0.5500   
                        10                   0.8309  0.7801  0.7292  0.6272   
                        21                   0.7783  0.7438  0.7092  0.6398   

transaction_cost_bps                           50.0  
portfolio               rebalance_frequency          
Composite Score         1                   -0.5553  
                        5                    0.1817  
                        10                   0.4017  
                        21                   0.5562  
Fixed 50/50 Sleeves     1                   -0.6065  
                        5                    0.1373  
                        10                   0.3075  
                        21                   0.4092  
Pure Inverse Volatility 1                   -0.7016  
                        5                    0.1194  
                        10                   0.3213  
                        21                   0.4313

transaction_cost_bps                           0.0     5.0     10.0    20.0  \
portfolio               rebalance_frequency                                   
Composite Score         1                    0.8401  0.7006  0.5610  0.2819   
                        5                    0.7443  0.6820  0.6197  0.4950   
                        10                   0.7902  0.7456  0.7009  0.6114   
                        21                   0.7608  0.7311  0.7013  0.6417   
Fixed 50/50 Sleeves     1                    0.8300  0.6864  0.5428  0.2555   
                        5                    0.7795  0.7138  0.6481  0.5166   
                        10                   0.7172  0.6705  0.6238  0.5305   
                        21                   0.6784  0.6467  0.6150  0.5515   
Pure Inverse Volatility 1                    0.8667  0.7099  0.5531  0.2394   
                        5                    0.8228  0.7516  0.6803  0.5375   
                        10                   0.7633  0.7121  0.6608  0.5582   
                        21                   0.7138  0.6789  0.6440  0.5741   

transaction_cost_bps                           50.0  
portfolio               rebalance_frequency          
Composite Score         1                   -0.5553  
                        5                    0.1217  
                        10                   0.3426  
                        21                   0.4620  
Fixed 50/50 Sleeves     1                   -0.6065  
                        5                    0.1225  
                        10                   0.2508  
                        21                   0.3610  
Pure Inverse Volatility 1                   -0.7016  
                        5                    0.1088  
                        10                   0.2510  
                        21                   0.3643

### Joint transaction-cost and rebalance-phase findings

Combining transaction-cost and rebalance-offset sensitivity confirms that the robust implementation region is approximately 10–21 trading days. The earlier conclusions are not dependent on offset 0.

#### Composite Score

Composite Score exhibits the clearest preference for the 21-day frequency.

At the baseline 10-bps cost:

- phase-averaged annualised return is 15.84% at 21 days, compared with 14.83% at 10 days;
- phase-averaged Sharpe is 0.802, compared with 0.759;
- the worst-offset Sharpe is 0.701, effectively identical to the 10-day worst-offset Sharpe of 0.701.

The 21-day advantage becomes larger as costs increase. At 50 bps, the phase-averaged 21-day implementation still earns 10.00% annually with a 0.556 Sharpe ratio, while its worst offset retains a 0.462 Sharpe ratio.

Composite Score therefore benefits from both lower turnover and slower portfolio refresh without showing material average signal decay.

#### Fixed 50/50 Sleeves

At 10 bps, the 10-day frequency provides the strongest central result:

- phase-averaged return is 10.50%, compared with 10.20% at 5 days and 10.16% at 21 days;
- phase-averaged Sharpe is 0.680, compared with 0.661 and 0.663 respectively.

The 5-day frequency has the highest worst-offset Sharpe at the baseline cost, although its average performance is lower. This comparison should be interpreted cautiously because longer frequencies have more possible offsets and therefore more opportunities to produce an extreme minimum.

As transaction costs increase, the 21-day implementation becomes preferable. At 20 bps, its phase-averaged Sharpe is 0.599, compared with 0.587 at 10 days. At 50 bps, the difference widens to 0.409 versus 0.308.

The cost-grid crossover between the 10- and 21-day variants occurs approximately in the 15–17 bps region. This is a historical sensitivity result rather than a precisely estimated trading-cost threshold.

#### Pure Inverse Volatility

Pure Inverse Volatility follows a similar pattern.

At 10 bps:

- the 10-day implementation earns 11.10% annually with a 0.729 phase-averaged Sharpe;
- the 21-day implementation earns 10.76% with a 0.709 Sharpe; and
- the 5-day implementation earns 10.48% with a 0.694 Sharpe.

The 21-day portfolio becomes stronger under more conservative cost assumptions. At 20 bps, its phase-averaged Sharpe is 0.640, compared with 0.627 at 10 days. At 50 bps, the respective values are 0.431 and 0.321.

The approximate crossover again lies between 15 and 17 bps.

#### Overall conclusion

Daily rebalancing is rejected because its modest gross-performance benefit, where present, is overwhelmed by turnover costs.

The robust implementation candidates are:

- 21 days for Composite Score;
- 10 days for the sleeve portfolios under the baseline 10-bps assumption; and
- 21 days for the sleeve portfolios when adopting a materially more conservative cost assumption.

These results identify a stable implementation region rather than a universally optimal frequency. The original 5-day portfolios should remain the pre-specified research baseline, while the 10- and 21-day variants are reported as implementation-robustness alternatives. Retrospectively choosing a different frequency for each portfolio would otherwise introduce an additional layer of in-sample selection.

Phase-averaged statistics describe sensitivity across alternative schedules; they are not the performance of a directly traded phase-averaged portfolio. Likewise, worst-offset results are operational stress measures rather than statistical confidence bounds.

### Subperiod stability

In [23]:
IMPLEMENTATION_SUBPERIODS = {
    "2016–2018": (
        pd.Timestamp("2016-01-07"),
        pd.Timestamp("2018-12-31"),
    ),
    "2019–2022": (
        pd.Timestamp("2019-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "2023–present": (
        pd.Timestamp("2023-01-01"),
        robustness_reference_dates.max(),
    ),
}

implementation_subperiod_rows = []

for subperiod, (start_date, end_date) in IMPLEMENTATION_SUBPERIODS.items():
    period_daily = rebalance_offset_daily.loc[
        rebalance_offset_daily["date"].between(
            start_date,
            end_date,
        )
    ].copy()

    for group_values, daily in period_daily.groupby(
        [
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
        ],
        sort=True,
    ):
        (
            portfolio_name,
            rebalance_frequency,
            rebalance_offset,
        ) = group_values

        net_summary = summarise_backtest(
            daily,
            return_column="net_return",
        ).iloc[0]

        implementation_subperiod_rows.append(
            {
                "subperiod": subperiod,
                "portfolio": portfolio_name,
                "rebalance_frequency": (rebalance_frequency),
                "rebalance_offset": rebalance_offset,
                "observations": len(daily),
                "net_annualised_return": (net_summary["annualised_return"]),
                "net_annualised_volatility": (net_summary["annualised_volatility"]),
                "net_sharpe": (net_summary["sharpe_ratio"]),
                "net_max_drawdown": (net_summary["max_drawdown"]),
                "average_daily_turnover": (net_summary["average_daily_turnover"]),
            }
        )

implementation_subperiod_summary = pd.DataFrame(implementation_subperiod_rows)

expected_subperiod_variants = (
    len(IMPLEMENTATION_SUBPERIODS) * len(ACTIVE_PORTFOLIOS) * sum(REBALANCE_FREQUENCIES)
)

if len(implementation_subperiod_summary) != expected_subperiod_variants:
    raise ValueError(
        f"Expected {expected_subperiod_variants} "
        "subperiod variants, found "
        f"{len(implementation_subperiod_summary)}."
    )

In [24]:
implementation_subperiod_range_summary = implementation_subperiod_summary.groupby(
    [
        "subperiod",
        "portfolio",
        "rebalance_frequency",
    ],
    sort=False,
).agg(
    offset_count=(
        "rebalance_offset",
        "nunique",
    ),
    mean_net_annualised_return=(
        "net_annualised_return",
        "mean",
    ),
    median_net_annualised_return=(
        "net_annualised_return",
        "median",
    ),
    minimum_net_annualised_return=(
        "net_annualised_return",
        "min",
    ),
    maximum_net_annualised_return=(
        "net_annualised_return",
        "max",
    ),
    mean_net_sharpe=(
        "net_sharpe",
        "mean",
    ),
    median_net_sharpe=(
        "net_sharpe",
        "median",
    ),
    minimum_net_sharpe=(
        "net_sharpe",
        "min",
    ),
    maximum_net_sharpe=(
        "net_sharpe",
        "max",
    ),
    worst_max_drawdown=(
        "net_max_drawdown",
        "min",
    ),
    mean_daily_turnover=(
        "average_daily_turnover",
        "mean",
    ),
)

In [25]:
phase_averaged_subperiod_return = implementation_subperiod_range_summary[
    "mean_net_annualised_return"
].unstack("subperiod")

phase_averaged_subperiod_sharpe = implementation_subperiod_range_summary[
    "mean_net_sharpe"
].unstack("subperiod")

worst_offset_subperiod_sharpe = implementation_subperiod_range_summary[
    "minimum_net_sharpe"
].unstack("subperiod")

display(phase_averaged_subperiod_return.round(4))

display(phase_averaged_subperiod_sharpe.round(4))

display(worst_offset_subperiod_sharpe.round(4))

subperiod                                    2016–2018  2019–2022  \
portfolio               rebalance_frequency                         
Composite Score         1                       0.0103    -0.0258   
                        5                       0.0387     0.0031   
                        10                      0.0578     0.0177   
                        21                      0.0765     0.0286   
Fixed 50/50 Sleeves     1                       0.0137    -0.0039   
                        5                       0.0265     0.0159   
                        10                      0.0359     0.0167   
                        21                      0.0376     0.0126   
Pure Inverse Volatility 1                       0.0071     0.0165   
                        5                       0.0193     0.0468   
                        10                      0.0305     0.0532   
                        21                      0.0302     0.0492   

subperiod                                    2023–present  
portfolio               rebalance_frequency                
Composite Score         1                          0.3663  
                        5                          0.3943  
                        10                         0.4160  
                        21                         0.4145  
Fixed 50/50 Sleeves     1                          0.2521  
                        5                          0.2862  
                        10                         0.2856  
                        21                         0.2779  
Pure Inverse Volatility 1                          0.2280  
                        5                          0.2596  
                        10                         0.2602  
                        21                         0.2548

subperiod                                    2016–2018  2019–2022  \
portfolio               rebalance_frequency                         
Composite Score         1                       0.1455    -0.0276   
                        5                       0.3081     0.1166   
                        10                      0.4157     0.1870   
                        21                      0.5207     0.2409   
Fixed 50/50 Sleeves     1                       0.1680     0.0527   
                        5                       0.2597     0.1793   
                        10                      0.3266     0.1845   
                        21                      0.3405     0.1582   
Pure Inverse Volatility 1                       0.1200     0.1865   
                        5                       0.2090     0.3927   
                        10                      0.2895     0.4383   
                        21                      0.2879     0.4131   

subperiod                                    2023–present  
portfolio               rebalance_frequency                
Composite Score         1                          1.3590  
                        5                          1.4348  
                        10                         1.4953  
                        21                         1.4845  
Fixed 50/50 Sleeves     1                          1.2084  
                        5                          1.3348  
                        10                         1.3320  
                        21                         1.2974  
Pure Inverse Volatility 1                          1.1264  
                        5                          1.2457  
                        10                         1.2461  
                        21                         1.2176

subperiod                                    2016–2018  2019–2022  \
portfolio               rebalance_frequency                         
Composite Score         1                       0.1455    -0.0276   
                        5                       0.2249     0.0544   
                        10                      0.3404     0.0658   
                        21                      0.3842     0.0494   
Fixed 50/50 Sleeves     1                       0.1680     0.0527   
                        5                       0.2133     0.1553   
                        10                      0.2524     0.0715   
                        21                      0.1767     0.0148   
Pure Inverse Volatility 1                       0.1200     0.1865   
                        5                       0.1711     0.3369   
                        10                      0.2032     0.3652   
                        21                      0.1242     0.1800   

subperiod                                    2023–present  
portfolio               rebalance_frequency                
Composite Score         1                          1.3590  
                        5                          1.3045  
                        10                         1.3872  
                        21                         1.3883  
Fixed 50/50 Sleeves     1                          1.2084  
                        5                          1.2436  
                        10                         1.2576  
                        21                         1.1268  
Pure Inverse Volatility 1                          1.1264  
                        5                          1.1576  
                        10                         1.1680  
                        21                         1.0366

### Subperiod stability findings

The subperiod analysis shows that slower rebalancing generally improves net performance, but the magnitude and preferred frequency vary through time.

#### Composite Score

Composite Score benefits substantially from slower rebalancing during 2016–2018:

- phase-averaged Sharpe increases from 0.146 with daily rebalancing to 0.416 at 10 days and 0.521 at 21 days;
- the worst-offset Sharpe also improves monotonically, reaching 0.384 at 21 days.

The portfolio is much weaker during 2019–2022. Daily rebalancing produces a negative phase-averaged return, while the 21-day implementation earns 2.86% annually with a 0.241 Sharpe ratio. However, its worst-offset Sharpe is only 0.049, showing that some monthly phases are barely profitable.

During 2023–present, both 10- and 21-day implementations are exceptionally strong. Their phase-averaged Sharpe ratios are 1.495 and 1.485 respectively, with almost identical worst-offset performance.

The full-sample preference for 21 days is therefore credible but not universal. It is strongest during 2016–2018, while 10 days is marginally stronger during the recent period and more robust in the weak 2019–2022 regime.

#### Fixed 50/50 Sleeves

Fixed 50/50 does not exhibit a stable preference for 21-day rebalancing.

- During 2016–2018, the 21-day variant has the highest phase-averaged Sharpe, but its worst-offset Sharpe is materially below that of the 10-day variant.
- During 2019–2022, the 5- and 10-day variants have similar average Sharpe ratios, while 5 days has substantially stronger worst-offset performance.
- During 2023–present, 5 and 10 days again perform similarly and both exceed 21 days.

The evidence therefore favours a broad 5–10-day implementation region rather than monthly rebalancing. Ten days remains the strongest full-sample central estimate, while five days provides competitive temporal and phase robustness.

#### Pure Inverse Volatility

Pure Inverse Volatility provides the clearest frequency result. The 10-day implementation has the highest phase-averaged and worst-offset Sharpe in every subperiod:

- 2016–2018: mean 0.290 and worst offset 0.203;
- 2019–2022: mean 0.438 and worst offset 0.365;
- 2023–present: mean 1.246 and worst offset 1.168.

Although some differences are small, the consistency across both time periods and rebalance phases makes 10 days the most robust implementation candidate for this portfolio.

#### Regime dependence

Absolute performance varies much more across subperiods than it does across reasonable rebalance frequencies.

All portfolios perform exceptionally strongly during 2023–present, with phase-averaged Sharpe ratios above 1.1. Earlier results are materially weaker:

- Composite Score is particularly weak during 2019–2022;
- Fixed 50/50 produces only modest performance before 2023; and
- Pure Inverse Volatility is the strongest sleeve construction during 2019–2022 but still remains far below its recent performance.

The full-sample statistics therefore conceal substantial time concentration. Slower rebalancing improves implementation efficiency, but it does not eliminate dependence on the underlying factor regime.

#### Interim conclusion

The combined evidence supports:

- 21 days as a credible slower implementation for Composite Score, with 10 days retained as a more conservative alternative;
- 5–10 days for Fixed 50/50, without robust evidence for extending to 21 days; and
- 10 days for Pure Inverse Volatility, supported consistently across time and rebalance phases.

These remain robustness alternatives rather than retrospectively selected replacements for the original five-day research specification.

Worst-offset comparisons should again be interpreted cautiously because frequencies with more possible phases have more opportunities to produce an extreme minimum.

### Rolling-window stability

In [26]:
ROLLING_IMPLEMENTATION_WINDOW = 252
TRADING_DAYS_PER_YEAR = 252

rolling_implementation_parts = []

rolling_group_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
]

for group_values, daily in rebalance_offset_daily.groupby(
    rolling_group_columns,
    sort=True,
):
    (
        portfolio_name,
        rebalance_frequency,
        rebalance_offset,
    ) = group_values

    daily = daily.sort_values("date").reset_index(drop=True).copy()

    net_return = daily["net_return"].astype(float)

    rolling_log_growth = (
        np.log1p(net_return)
        .rolling(
            ROLLING_IMPLEMENTATION_WINDOW,
            min_periods=ROLLING_IMPLEMENTATION_WINDOW,
        )
        .sum()
    )

    rolling_annualised_return = np.expm1(
        rolling_log_growth * TRADING_DAYS_PER_YEAR / ROLLING_IMPLEMENTATION_WINDOW
    )

    rolling_annualised_volatility = net_return.rolling(
        ROLLING_IMPLEMENTATION_WINDOW,
        min_periods=ROLLING_IMPLEMENTATION_WINDOW,
    ).std(ddof=1) * np.sqrt(TRADING_DAYS_PER_YEAR)

    rolling_sharpe = rolling_annualised_return / rolling_annualised_volatility

    rolling_part = pd.DataFrame(
        {
            "date": daily["date"],
            "portfolio": portfolio_name,
            "rebalance_frequency": (rebalance_frequency),
            "rebalance_offset": rebalance_offset,
            "rolling_annualised_return": (rolling_annualised_return),
            "rolling_annualised_volatility": (rolling_annualised_volatility),
            "rolling_sharpe": rolling_sharpe,
        }
    )

    rolling_implementation_parts.append(rolling_part.dropna(subset=["rolling_sharpe"]))

rolling_implementation = pd.concat(
    rolling_implementation_parts,
    ignore_index=True,
)

In [27]:
rolling_phase_summary = (
    rolling_implementation.groupby(
        [
            "portfolio",
            "rebalance_frequency",
            "date",
        ],
        sort=True,
    )
    .agg(
        offset_count=(
            "rebalance_offset",
            "nunique",
        ),
        mean_rolling_annualised_return=(
            "rolling_annualised_return",
            "mean",
        ),
        mean_rolling_sharpe=(
            "rolling_sharpe",
            "mean",
        ),
        median_rolling_sharpe=(
            "rolling_sharpe",
            "median",
        ),
        minimum_rolling_sharpe=(
            "rolling_sharpe",
            "min",
        ),
        maximum_rolling_sharpe=(
            "rolling_sharpe",
            "max",
        ),
    )
    .reset_index()
)

offset_count_audit = (
    rolling_phase_summary["offset_count"]
    == rolling_phase_summary["rebalance_frequency"]
)

if not offset_count_audit.all():
    raise ValueError(
        "At least one rolling date has an " "unexpected number of offsets."
    )

In [28]:
rolling_frequency_stability = rolling_phase_summary.groupby(
    [
        "portfolio",
        "rebalance_frequency",
    ]
).agg(
    rolling_windows=("date", "size"),
    median_phase_averaged_sharpe=(
        "mean_rolling_sharpe",
        "median",
    ),
    tenth_percentile_phase_averaged_sharpe=(
        "mean_rolling_sharpe",
        lambda values: values.quantile(0.10),
    ),
    minimum_phase_averaged_sharpe=(
        "mean_rolling_sharpe",
        "min",
    ),
    maximum_phase_averaged_sharpe=(
        "mean_rolling_sharpe",
        "max",
    ),
    positive_phase_averaged_fraction=(
        "mean_rolling_sharpe",
        lambda values: (values > 0).mean(),
    ),
    median_worst_offset_sharpe=(
        "minimum_rolling_sharpe",
        "median",
    ),
    positive_worst_offset_fraction=(
        "minimum_rolling_sharpe",
        lambda values: (values > 0).mean(),
    ),
)

In [29]:
rolling_frequency_winners = rolling_phase_summary.loc[
    rolling_phase_summary.groupby(["portfolio", "date"])[
        "mean_rolling_sharpe"
    ].idxmax(),
    [
        "portfolio",
        "date",
        "rebalance_frequency",
    ],
]

rolling_frequency_win_fraction = (
    rolling_frequency_winners.groupby(
        [
            "portfolio",
            "rebalance_frequency",
        ]
    )
    .size()
    .div(
        rolling_frequency_winners.groupby("portfolio").size(),
        level="portfolio",
    )
    .rename("winning_window_fraction")
    .unstack(
        "rebalance_frequency",
        fill_value=0.0,
    )
)

In [30]:
display(rolling_frequency_stability.round(4))

display(rolling_frequency_win_fraction.round(4))

rolling_windows  \
portfolio               rebalance_frequency                    
Composite Score         1                               2384   
                        5                               2384   
                        10                              2384   
                        21                              2384   
Fixed 50/50 Sleeves     1                               2384   
                        5                               2384   
                        10                              2384   
                        21                              2384   
Pure Inverse Volatility 1                               2384   
                        5                               2384   
                        10                              2384   
                        21                              2384   

                                             median_phase_averaged_sharpe  \
portfolio               rebalance_frequency                                 
Composite Score         1                                          0.5884   
                        5                                          0.7719   
                        10                                         0.8564   
                        21                                         0.9136   
Fixed 50/50 Sleeves     1                                          0.5863   
                        5                                          0.7620   
                        10                                         0.7880   
                        21                                         0.8200   
Pure Inverse Volatility 1                                          0.5590   
                        5                                          0.7555   
                        10                                         0.8007   
                        21                                         0.8255   

                                             tenth_percentile_phase_averaged_sharpe  \
portfolio               rebalance_frequency                                           
Composite Score         1                                                   -0.7825   
                        5                                                   -0.7545   
                        10                                                  -0.6483   
                        21                                                  -0.5781   
Fixed 50/50 Sleeves     1                                                   -0.7537   
                        5                                                   -0.6224   
                        10                                                  -0.6228   
                        21                                                  -0.6114   
Pure Inverse Volatility 1                                                   -0.7080   
                        5                                                   -0.4778   
                        10                                                  -0.4791   
                        21                                                  -0.5144   

                                             minimum_phase_averaged_sharpe  \
portfolio               rebalance_frequency                                  
Composite Score         1                                          -1.2686   
                        5                                          -1.2943   
                        10                                         -1.2662   
                        21                                         -1.2451   
Fixed 50/50 Sleeves     1                                          -1.3103   
                        5                                          -1.2458   
                        10                                         -1.3175   
                        21                                         -1.3693   
Pure Inverse Volatility 1                                          -1.

rebalance_frequency,1,5,10,21
portfolio,,,,
Composite Score,0.0453,0.0344,0.1468,0.7735
Fixed 50/50 Sleeves,0.0235,0.2047,0.4220,0.3498
Pure Inverse Volatility,0.0260,0.1812,0.4673,0.3255


### Rolling-window frequency stability

The rolling 252-day analysis confirms that the preferred implementation frequencies are not determined solely by full-sample averages. However, no frequency dominates universally, and frequency differences remain smaller than the variation in performance through time.

#### Composite Score

Composite Score provides the clearest evidence in favour of 21-day rebalancing.

As the rebalance interval increases from 5 to 21 trading days:

- median phase-averaged rolling Sharpe rises from 0.772 to 0.914;
- the 10th-percentile Sharpe improves from -0.755 to -0.578;
- the fraction of positive rolling windows increases from 72.6% to 75.6%; and
- the fraction in which even the worst offset is profitable increases from 70.4% to 70.7%.

Most decisively, the 21-day frequency has the highest phase-averaged rolling Sharpe in 77.4% of all windows. The 10-day version wins only 14.7%, while the daily and 5-day variants rarely lead.

The 21-day improvement is therefore not confined to one subperiod or a favourable rebalance phase. It raises typical performance while modestly improving downside rolling statistics. This supports 21 days as the principal implementation candidate for Composite Score.

#### Fixed 50/50 Sleeves

For Fixed 50/50, the evidence is more balanced.

Median phase-averaged rolling Sharpe increases from 0.762 at 5 days to 0.788 at 10 days and 0.820 at 21 days. However, the 10-day implementation wins more frequently:

- 5 days: 20.5% of rolling windows;
- 10 days: 42.2%;
- 21 days: 35.0%.

The 21-day frequency is also more sensitive to rebalance phase. Its median worst-offset Sharpe is only 0.464, compared with 0.599 at 10 days and 0.664 at 5 days. The worst offset remains profitable in 63.9% of rolling windows at 21 days, versus 67.0% at 10 days and 68.5% at 5 days.

Thus, 21 days produces the strongest median result but does not dominate consistently. Ten days offers the best balance between typical performance, frequency leadership, transaction-cost reduction, and phase robustness. Five days remains a defensible alternative when robustness to implementation timing is prioritised.

#### Pure Inverse Volatility

Pure Inverse Volatility produces a similar but clearer result.

The 21-day frequency has the highest median phase-averaged Sharpe, at 0.826, compared with 0.801 at 10 days. Nevertheless, the 10-day frequency:

- wins the largest fraction of rolling windows, at 46.7%;
- has the highest positive phase-averaged fraction, at 75.2%;
- has a median worst-offset Sharpe of 0.598, compared with 0.451 at 21 days; and
- keeps the worst offset profitable in 71.4% of windows, compared with 68.1% at 21 days.

The 5-day version provides the strongest worst-offset statistics but wins only 18.1% of windows. Ten days therefore represents the most balanced implementation: it captures most of the benefit from slower rebalancing without taking on the greater phase sensitivity of the 21-day schedule.

#### Final frequency conclusion

Combining full-sample performance, transaction-cost sensitivity, rebalance offsets, subperiods, and rolling windows supports the following implementation candidates:

- **Composite Score: 21 trading days**
- **Fixed 50/50 Sleeves: 10 trading days**
- **Pure Inverse Volatility: 10 trading days**

The original 5-day variants remain the pre-specified research baselines. They should continue to be reported so that the implementation choices are not presented as retrospectively selected replacements.

For the sleeve portfolios, 5–10 days forms a generally stable implementation region. Five days provides stronger protection against unfavourable rebalance phases, while ten days offers the best overall balance between average performance and turnover.

Absolute performance remains strongly time-varying. Negative 10th-percentile rolling Sharpe ratios and minimum rolling Sharpe ratios below -1 show that none of the implementations eliminates prolonged weak periods.

The rolling windows overlap substantially, and the phase variants are evaluated on the same underlying returns. Win fractions and rolling distributions are therefore descriptive stability diagnostics, not independent observations or statistical significance measures.

### Turnover concentration analysis

Slower portfolios may trade less frequently but make larger adjustments on each rebalance date.

#### Summarise turnover for every phase

In [31]:
TURNOVER_TOLERANCE = 1e-12


def calculate_top_n_turnover_share(
    turnover,
    number_of_days,
):
    turnover = turnover.clip(lower=0.0)
    total_turnover = turnover.sum()

    if total_turnover <= TURNOVER_TOLERANCE:
        return np.nan

    return turnover.nlargest(number_of_days).sum() / total_turnover


turnover_concentration_rows = []

turnover_group_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
]

for group_values, daily in rebalance_offset_daily.groupby(
    turnover_group_columns,
    sort=True,
):
    (
        portfolio_name,
        rebalance_frequency,
        rebalance_offset,
    ) = group_values

    daily = daily.sort_values("date").reset_index(drop=True).copy()

    turnover = daily["turnover"].astype(float).clip(lower=0.0)

    active_turnover = turnover.loc[turnover > TURNOVER_TOLERANCE]

    total_turnover = turnover.sum()
    sample_years = len(turnover) / 252.0

    turnover_concentration_rows.append(
        {
            "portfolio": portfolio_name,
            "rebalance_frequency": (rebalance_frequency),
            "rebalance_offset": rebalance_offset,
            "observations": len(turnover),
            "trading_days": len(active_turnover),
            "trading_day_fraction": (len(active_turnover) / len(turnover)),
            "mean_daily_turnover": turnover.mean(),
            "annualised_turnover": (total_turnover / sample_years),
            "mean_rebalance_turnover": (active_turnover.mean()),
            "median_rebalance_turnover": (active_turnover.median()),
            "p90_rebalance_turnover": (active_turnover.quantile(0.90)),
            "p95_rebalance_turnover": (active_turnover.quantile(0.95)),
            "p99_rebalance_turnover": (active_turnover.quantile(0.99)),
            "maximum_daily_turnover": (turnover.max()),
            "maximum_21_day_turnover": (
                turnover.rolling(
                    21,
                    min_periods=1,
                )
                .sum()
                .max()
            ),
            "top_1_day_turnover_share": (
                calculate_top_n_turnover_share(
                    turnover,
                    1,
                )
            ),
            "top_5_day_turnover_share": (
                calculate_top_n_turnover_share(
                    turnover,
                    5,
                )
            ),
            "top_10_day_turnover_share": (
                calculate_top_n_turnover_share(
                    turnover,
                    10,
                )
            ),
        }
    )

turnover_concentration_by_offset = (
    pd.DataFrame(turnover_concentration_rows)
    .sort_values(turnover_group_columns)
    .reset_index(drop=True)
)

In [32]:
turnover_concentration_summary = (
    turnover_concentration_by_offset.groupby(
        [
            "portfolio",
            "rebalance_frequency",
        ]
    )
    .agg(
        offset_count=(
            "rebalance_offset",
            "nunique",
        ),
        mean_daily_turnover=(
            "mean_daily_turnover",
            "mean",
        ),
        mean_annualised_turnover=(
            "annualised_turnover",
            "mean",
        ),
        median_rebalance_turnover=(
            "median_rebalance_turnover",
            "median",
        ),
        median_p95_rebalance_turnover=(
            "p95_rebalance_turnover",
            "median",
        ),
        worst_phase_p95_rebalance_turnover=(
            "p95_rebalance_turnover",
            "max",
        ),
        worst_single_day_turnover=(
            "maximum_daily_turnover",
            "max",
        ),
        median_maximum_21_day_turnover=(
            "maximum_21_day_turnover",
            "median",
        ),
        worst_maximum_21_day_turnover=(
            "maximum_21_day_turnover",
            "max",
        ),
        median_top_5_day_turnover_share=(
            "top_5_day_turnover_share",
            "median",
        ),
        maximum_top_5_day_turnover_share=(
            "top_5_day_turnover_share",
            "max",
        ),
    )
)

In [33]:
turnover_offset_count_audit = turnover_concentration_summary[
    "offset_count"
] == turnover_concentration_summary.index.get_level_values("rebalance_frequency")

if not turnover_offset_count_audit.all():
    raise ValueError(
        "Unexpected number of rebalance offsets " "in the turnover summary."
    )

In [34]:
IMPLEMENTATION_FREQUENCIES = {
    "Composite Score": 21,
    "Fixed 50/50 Sleeves": 10,
    "Pure Inverse Volatility": 10,
}

implementation_turnover_rows = [
    (portfolio, frequency)
    for portfolio, frequency in IMPLEMENTATION_FREQUENCIES.items()
]

implementation_turnover_summary = turnover_concentration_summary.loc[
    implementation_turnover_rows
]

In [35]:
display(turnover_concentration_summary.round(4))

display(implementation_turnover_summary.round(4))

offset_count  \
portfolio               rebalance_frequency                 
Composite Score         1                               1   
                        5                               5   
                        10                             10   
                        21                             21   
Fixed 50/50 Sleeves     1                               1   
                        5                               5   
                        10                             10   
                        21                             21   
Pure Inverse Volatility 1                               1   
                        5                               5   
                        10                             10   
                        21                             21   

                                             mean_daily_turnover  \
portfolio               rebalance_frequency                        
Composite Score         1                                 0.2359   
                        5                                 0.1055   
                        10                                0.0748   
                        21                                0.0511   
Fixed 50/50 Sleeves     1                                 0.1919   
                        5                                 0.0875   
                        10                                0.0617   
                        21                                0.0418   
Pure Inverse Volatility 1                                 0.2021   
                        5                                 0.0927   
                        10                                0.0655   
                        21                                0.0446   

                                             mean_annualised_turnover  \
portfolio               rebalance_frequency                             
Composite Score         1                                     59.4524   
                        5                                     26.5918   
                        10                                    18.8383   
                        21                                    12.8750   
Fixed 50/50 Sleeves     1                                     48.3615   
                        5                                     22.0404   
                        10                                    15.5540   
                        21                                    10.5249   
Pure Inverse Volatility 1                                     50.9234   
                        5                                     23.3536   
                        10                                    16.5153   
                        21                                    11.2370   

                                             median_rebalance_turnover  \
portfolio               rebalance_frequency                              
Composite Score         1                                       0.2230   
                        5                                       0.5233   
                        10                                      0.7350   
                        21                                      1.0558   
Fixed 50/50 Sleeves     1                                       0.1769   
                        5                                       0.4271   
                        10                                      0.6058   
                        21                                      0.8659   
Pure Inverse Volatility 1                                       0.1947   
                        5                                       0.4501   
                        10                                      0.6362   
                        21                                      0.9259   

                                             median_p95_rebalance_turnover  \
portfolio               rebalance_frequency                                  
Composite Score      

,,offset_count,mean_daily_turnover,mean_annualised_turnover,median_rebalance_turnover,median_p95_rebalance_turnover,worst_phase_p95_rebalance_turnover,worst_single_day_turnover,median_maximum_21_day_turnover,worst_maximum_21_day_turnover,median_top_5_day_turnover_share,maximum_top_5_day_turnover_share
portfolio,rebalance_frequency,,,,,,,,,,,
Composite Score,21,21,0.0511,12.8750,1.0558,1.5619,1.6741,2.0860,2.0000,2.0860,0.0678,0.0715
Fixed 50/50 Sleeves,10,10,0.0617,15.5540,0.6058,0.8892,0.9359,1.3304,2.9269,3.2722,0.0338,0.0348
Pure Inverse Volatility,10,10,0.0655,16.5153,0.6362,0.9458,0.9871,1.3304,3.0111,3.2851,0.0323,0.0336


### Turnover concentration findings

Slower rebalancing materially reduces total trading, but it concentrates portfolio adjustment into larger rebalance events. The selected frequencies therefore improve cost efficiency without uniformly reducing every dimension of implementation risk.

#### Composite Score: 21 days

The 21-day Composite Score implementation has the lowest aggregate turnover among the selected candidates:

- mean daily turnover: 5.11%;
- annualised turnover: 12.88 times portfolio capital;
- median rebalance turnover: 1.056;
- median phase-level 95th-percentile rebalance turnover: 1.562; and
- worst single-day turnover: 2.086.

Relative to the original 5-day specification, annualised turnover falls by approximately 52%. Relative to daily rebalancing, it falls by approximately 78%.

However, this reduction comes through infrequent, large portfolio transitions. A typical rebalance trades approximately 1.06 units of notional per unit of portfolio capital under the current turnover convention. The 95th-percentile event is approximately 1.56, and the most extreme historical event exceeds 2.0.

Composite Score therefore has the lowest cumulative trading burden but the greatest single-day execution requirement.

#### Fixed 50/50 Sleeves: 10 days

The selected 10-day Fixed 50/50 implementation has:

- mean daily turnover: 6.17%;
- annualised turnover: 15.55;
- median rebalance turnover: 0.606;
- median 95th-percentile rebalance turnover: 0.889; and
- worst single-day turnover: 1.330.

Annualised turnover is approximately 29% below the 5-day specification and 68% below daily rebalancing.

Its median rebalance is approximately 43% smaller than that of the 21-day Composite portfolio. However, a rolling 21-day window can contain as many as three 10-day rebalances, producing a median phase-level maximum cumulative turnover of 2.927 and a worst value of 3.272.

#### Pure Inverse Volatility: 10 days

Pure Inverse Volatility has a similar profile:

- mean daily turnover: 6.55%;
- annualised turnover: 16.52;
- median rebalance turnover: 0.636;
- median 95th-percentile rebalance turnover: 0.946; and
- worst single-day turnover: 1.330.

Annualised turnover is approximately 29% below the 5-day version and 68% below daily rebalancing. It has the highest aggregate turnover of the selected implementations, although its individual rebalance events remain materially smaller than those of Composite Score.

Its worst rolling 21-day turnover reaches 3.285, reflecting the cumulative burden of multiple rebalances rather than one exceptional trading day.

#### Concentration across dates

The five largest trading days account for:

- approximately 6.8%–7.2% of total turnover for 21-day Composite Score;
- approximately 3.4%–3.5% for 10-day Fixed 50/50; and
- approximately 3.2%–3.4% for 10-day Pure Inverse Volatility.

Turnover is therefore not dominated by a very small number of historical events. The higher share for Composite is partly mechanical because it has fewer rebalance dates.

#### Implementation conclusion

The selected frequencies reduce total turnover rather than merely shifting the same amount of trading across time. Nevertheless, they expose two different liquidity stresses:

- Composite Score has the lowest annual trading burden but the largest same-day portfolio transition;
- the sleeve portfolios have smaller individual rebalances but greater cumulative turnover across multi-rebalance monthly windows.

Average turnover alone is therefore insufficient for assessing capacity. Capacity must be evaluated using security-level trade weights and lagged dollar volume. In particular, Composite Score may encounter a same-day participation constraint before its lower annual turnover becomes the limiting consideration.

The current backtest assumes immediate execution at each rebalance. Any proposal to spread a large rebalance across several days would require a separate execution-delay and signal-decay test.

## Security-level capacity analysis

$$
\text{Participation}_{i,t}(A)
=
\frac{A |\Delta w_{i,t}|}
{\operatorname{ADV}^{21}_{i,t-1}},
$$

For a participation limit $q$,

$$
A^{\max}_{t}(q)
=
\min_i
\frac{q \operatorname{ADV}^{21}_{i,t-1}}
{|\Delta w_{i,t}|}.
$$

#### Select the implementation variants and audit turnover

In [36]:
implementation_frequency_table = (
    pd.Series(
        IMPLEMENTATION_FREQUENCIES,
        name="rebalance_frequency",
    )
    .rename_axis("portfolio")
    .reset_index()
)

implementation_security_trades = security_trade_detail.merge(
    implementation_frequency_table,
    on=[
        "portfolio",
        "rebalance_frequency",
    ],
    how="inner",
    validate="many_to_one",
).copy()

implementation_security_trades["date"] = pd.to_datetime(
    implementation_security_trades["date"]
)

implementation_security_trades["absolute_trade_weight"] = (
    implementation_security_trades["trade_weight"].astype(float).abs()
)

security_level_turnover = (
    implementation_security_trades.groupby(
        [
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
            "date",
        ],
        sort=True,
    )["absolute_trade_weight"]
    .sum()
    .rename("security_level_turnover")
    .reset_index()
)

daily_turnover_reference = rebalance_offset_daily.merge(
    implementation_frequency_table,
    on=[
        "portfolio",
        "rebalance_frequency",
    ],
    how="inner",
    validate="many_to_one",
)[
    [
        "portfolio",
        "rebalance_frequency",
        "rebalance_offset",
        "date",
        "turnover",
    ]
].copy()

turnover_audit = daily_turnover_reference.merge(
    security_level_turnover,
    on=[
        "portfolio",
        "rebalance_frequency",
        "rebalance_offset",
        "date",
    ],
    how="left",
    validate="one_to_one",
)

turnover_audit["security_level_turnover"] = turnover_audit[
    "security_level_turnover"
].fillna(0.0)

maximum_security_turnover_difference = (
    (turnover_audit["turnover"] - turnover_audit["security_level_turnover"]).abs().max()
)

if maximum_security_turnover_difference >= 1e-12:
    raise ValueError(
        "Security-level trades do not reproduce "
        "aggregate turnover. Check whether the engine "
        "uses full-L1 or half-L1 turnover and whether "
        "trade weights are measured against drifted "
        "pre-trade weights."
    )

print(
    "Maximum security-turnover difference:",
    maximum_security_turnover_difference,
)

Maximum security-turnover difference: 4.440892098500626e-16


#### Construct lagged dollar volume

In [37]:
security_market_data = (
    panel[
        [
            "date",
            "ticker",
            "close",
            "volume",
        ]
    ]
    .assign(date=lambda df: pd.to_datetime(df["date"]))
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

In [38]:
ADV_WINDOW = 21

security_liquidity = security_market_data[
    [
        "date",
        "ticker",
        "close",
        "volume",
    ]
].copy()

security_liquidity["date"] = pd.to_datetime(security_liquidity["date"])

security_liquidity = security_liquidity.sort_values(["ticker", "date"])

if security_liquidity.duplicated(["ticker", "date"]).any():
    raise ValueError("Duplicate ticker-date liquidity rows found.")

security_liquidity["dollar_volume"] = (
    security_liquidity["close"] * security_liquidity["volume"]
)

security_liquidity["lagged_adv_21"] = security_liquidity.groupby("ticker")[
    "dollar_volume"
].transform(
    lambda values: (
        values.rolling(
            ADV_WINDOW,
            min_periods=ADV_WINDOW,
        )
        .mean()
        .shift(1)
    )
)

#### Calculate daily capacity

In [39]:
PARTICIPATION_LIMITS = (0.01, 0.05, 0.10)
TRADE_WEIGHT_TOLERANCE = 1e-12

trade_liquidity = implementation_security_trades.merge(
    security_liquidity[
        [
            "date",
            "ticker",
            "lagged_adv_21",
        ]
    ],
    on=["date", "ticker"],
    how="left",
    validate="many_to_one",
)

trade_liquidity = trade_liquidity.loc[
    trade_liquidity["absolute_trade_weight"] > TRADE_WEIGHT_TOLERANCE
].copy()

capacity_daily_rows = []

capacity_group_columns = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
    "date",
]

for group_values, trades in trade_liquidity.groupby(
    capacity_group_columns,
    sort=True,
):
    (
        portfolio_name,
        rebalance_frequency,
        rebalance_offset,
        date,
    ) = group_values

    adv_is_valid = trades["lagged_adv_21"].notna() & (trades["lagged_adv_21"] > 0.0)

    fully_covered = bool(adv_is_valid.all())

    trade_share = (
        trades["absolute_trade_weight"] / trades["absolute_trade_weight"].sum()
    )

    common_values = {
        "portfolio": portfolio_name,
        "rebalance_frequency": (rebalance_frequency),
        "rebalance_offset": rebalance_offset,
        "date": date,
        "traded_security_count": len(trades),
        "fully_covered": fully_covered,
        "largest_security_trade_share": (trade_share.max()),
        "top_5_security_trade_share": (trade_share.nlargest(5).sum()),
        "effective_traded_security_count": (1.0 / trade_share.pow(2).sum()),
    }

    for participation_limit in PARTICIPATION_LIMITS:
        capacity = np.nan
        bottleneck_ticker = None

        if fully_covered:
            security_capacity = (
                participation_limit
                * trades["lagged_adv_21"]
                / trades["absolute_trade_weight"]
            )

            bottleneck_index = security_capacity.idxmin()

            capacity = security_capacity.loc[bottleneck_index]

            bottleneck_ticker = trades.loc[
                bottleneck_index,
                "ticker",
            ]

        capacity_daily_rows.append(
            {
                **common_values,
                "participation_limit": (participation_limit),
                "capacity_usd": capacity,
                "bottleneck_ticker": (bottleneck_ticker),
            }
        )

capacity_daily = pd.DataFrame(capacity_daily_rows)

#### Summarise across dates and offsets

In [40]:
capacity_by_offset = (
    capacity_daily.groupby(
        [
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
            "participation_limit",
        ],
        sort=True,
    )
    .agg(
        rebalance_days=("date", "nunique"),
        fully_covered_fraction=(
            "fully_covered",
            "mean",
        ),
        minimum_capacity_usd=(
            "capacity_usd",
            "min",
        ),
        fifth_percentile_capacity_usd=(
            "capacity_usd",
            lambda values: values.quantile(0.05),
        ),
        tenth_percentile_capacity_usd=(
            "capacity_usd",
            lambda values: values.quantile(0.10),
        ),
        median_capacity_usd=(
            "capacity_usd",
            "median",
        ),
        median_effective_traded_names=(
            "effective_traded_security_count",
            "median",
        ),
        maximum_largest_security_trade_share=(
            "largest_security_trade_share",
            "max",
        ),
        maximum_top_5_security_trade_share=(
            "top_5_security_trade_share",
            "max",
        ),
    )
    .reset_index()
)

capacity_phase_summary = capacity_by_offset.groupby(
    [
        "portfolio",
        "rebalance_frequency",
        "participation_limit",
    ],
    sort=True,
).agg(
    offset_count=(
        "rebalance_offset",
        "nunique",
    ),
    minimum_adv_coverage=(
        "fully_covered_fraction",
        "min",
    ),
    median_fifth_percentile_capacity_usd=(
        "fifth_percentile_capacity_usd",
        "median",
    ),
    worst_phase_fifth_percentile_capacity_usd=(
        "fifth_percentile_capacity_usd",
        "min",
    ),
    median_capacity_usd=(
        "median_capacity_usd",
        "median",
    ),
    worst_historical_capacity_usd=(
        "minimum_capacity_usd",
        "min",
    ),
)

capacity_phase_summary_millions = capacity_phase_summary.copy()

capacity_columns = [
    column
    for column in capacity_phase_summary_millions
    if column.endswith("_capacity_usd")
]

capacity_phase_summary_millions[capacity_columns] = (
    capacity_phase_summary_millions[capacity_columns] / 1_000_000.0
)

display(capacity_phase_summary_millions.round(2))

offset_count  \
portfolio               rebalance_frequency participation_limit                 
Composite Score         21                  0.01                           21   
                                            0.05                           21   
                                            0.10                           21   
Fixed 50/50 Sleeves     10                  0.01                           10   
                                            0.05                           10   
                                            0.10                           10   
Pure Inverse Volatility 10                  0.01                           10   
                                            0.05                           10   
                                            0.10                           10   

                                                                 minimum_adv_coverage  \
portfolio               rebalance_frequency participation_limit                         
Composite Score         21                  0.01                                  1.0   
                                            0.05                                  1.0   
                                            0.10                                  1.0   
Fixed 50/50 Sleeves     10                  0.01                                  1.0   
                                            0.05                                  1.0   
                                            0.10                                  1.0   
Pure Inverse Volatility 10                  0.01                                  1.0   
                                            0.05                                  1.0   
                                            0.10                                  1.0   

                                                                 median_fifth_percentile_capacity_usd  \
portfolio               rebalance_frequency participation_limit                                         
Composite Score         21                  0.01                                                27.42   
                                            0.05                                               137.12   
                                            0.10                                               274.23   
Fixed 50/50 Sleeves     10                  0.01                                                42.28   
                                            0.05                                               211.41   
                                            0.10                                               422.83   
Pure Inverse Volatility 10                  0.01                                                41.27   
                                            0.05                                               206.33   
                                            0.10                                               412.66   

                                                                 worst_phase_fifth_percentile_capacity_usd  \
portfolio               rebalance_frequency participation_limit                                              
Composite Score         21                  0.01                                                     24.77   
                                            0.05                                                    123.86   
                                            0.10                                                    247.72   
Fixed 50/50 Sleeves     10                  0.01                                                     40.17   
                                            0.05                                                    200.87   
                                            0.10                                                    401.75   
Pure Inverse Volatility 10                  0.01                                                     38.34   
                                            0.0

### Security-level capacity findings

The security-level capacity analysis confirms complete liquidity-data coverage and reveals a clear difference between the selected implementations.

All portfolios and rebalance phases have 100% valid lagged 21-day ADV coverage. The capacity estimates are therefore not being calculated from a selectively observed subset of trades.

#### Conservative 1% participation limit

At a 1% maximum participation rate, the worst-phase fifth-percentile capacity estimates are:

- Composite Score: $24.77 million;
- Fixed 50/50 Sleeves: $40.17 million; and
- Pure Inverse Volatility: $38.34 million.

These figures represent conservative, phase-robust capacity estimates: for even the least favourable rebalance phase, 95% of historical rebalances support at least the reported AUM under the assumed 1% ADV limit.

Relative to Composite Score, worst-phase fifth-percentile capacity is approximately:

- 62% higher for Fixed 50/50; and
- 55% higher for Pure Inverse Volatility.

The median fifth-percentile estimates tell the same story, at $27.42 million for Composite, $42.28 million for Fixed 50/50, and $41.27 million for Pure Inverse Volatility.

#### Typical capacity

Median capacity is materially higher than the conservative fifth-percentile measure:

- Composite Score: $52.42 million;
- Fixed 50/50 Sleeves: $96.90 million; and
- Pure Inverse Volatility: $92.80 million.

The sleeve portfolios therefore support approximately 1.8 times Composite's typical same-day capacity.

This confirms the hypothesis suggested by the turnover-concentration analysis. Although Composite has the lowest annual turnover, its 21-day schedule concentrates adjustment into larger same-day orders. The 10-day sleeve portfolios trade more cumulatively but generally place smaller security-level orders at each rebalance.

The difference may also reflect the portfolios' security composition and trade concentration, rather than rebalance frequency alone.

#### Participation-rate scenarios

Because capacity is proportional to the assumed participation limit, the 5% estimates are exactly five times the 1% estimates:

- Composite Score: $123.86 million;
- Fixed 50/50 Sleeves: $200.87 million; and
- Pure Inverse Volatility: $191.68 million,

using the worst-phase fifth percentile.

At a 10% limit, the corresponding estimates are $247.72 million, $401.75 million, and $383.36 million.

These rows should be interpreted as execution-assumption scenarios rather than independent evidence. A higher allowable participation rate increases theoretical capacity but would normally produce greater market impact and execution uncertainty.

#### Extreme historical events

The absolute historical minima differ from the more robust fifth-percentile results:

- Composite Score: $7.89 million;
- Fixed 50/50 Sleeves: $5.59 million; and
- Pure Inverse Volatility: $5.44 million.

Thus, while the sleeve portfolios have substantially greater capacity under normal and moderately adverse conditions, they experienced more severe isolated bottlenecks.

The absolute minimum may be driven by an exceptional trade, a temporarily illiquid security, an unusually large portfolio transition, or a market-data issue. It should be investigated but should not replace the fifth percentile as the primary capacity statistic.

#### Capacity conclusion

Under the conservative 1% participation assumption, approximate implementation-capacity ranges are:

- Composite Score: approximately $25 million;
- Fixed 50/50 Sleeves: approximately $40 million; and
- Pure Inverse Volatility: approximately $38 million.

These are liquidity-screening estimates rather than hard AUM limits. They assume that every rebalance is completed within one trading day and that participation in historical ADV is an adequate proxy for execution feasibility. They do not model bid–ask spreads, nonlinear market impact, volatility, intraday volume profiles, signal crowding, or liquidation requirements.

Before closing the capacity analysis, the recurring bottleneck securities and the extreme minimum-capacity events should be inspected. This will establish whether the lower-tail estimates represent genuine implementation risks or isolated data and portfolio-transition anomalies.

In [41]:
CAPACITY_DIAGNOSTIC_LIMIT = 0.01

capacity_diagnostic = (
    capacity_daily.loc[
        capacity_daily["participation_limit"].eq(
            CAPACITY_DIAGNOSTIC_LIMIT
        )
    ]
    .copy()
)

#### Identify recurring bottleneck securities

In [42]:
bottleneck_frequency = (
    capacity_diagnostic.groupby(
        [
            "portfolio",
            "rebalance_frequency",
            "bottleneck_ticker",
        ],
        sort=True,
    )
    .agg(
        bottleneck_observations=("date", "size"),
        median_capacity_usd=("capacity_usd", "median"),
        minimum_capacity_usd=("capacity_usd", "min"),
    )
    .reset_index()
)

bottleneck_frequency["bottleneck_fraction"] = bottleneck_frequency[
    "bottleneck_observations"
] / bottleneck_frequency.groupby(
    [
        "portfolio",
        "rebalance_frequency",
    ]
)[
    "bottleneck_observations"
].transform(
    "sum"
)

top_bottleneck_securities = (
    bottleneck_frequency.sort_values(
        [
            "portfolio",
            "rebalance_frequency",
            "bottleneck_observations",
        ],
        ascending=[True, True, False],
    )
    .groupby(
        [
            "portfolio",
            "rebalance_frequency",
        ],
        group_keys=False,
    )
    .head(10)
)

display(
    top_bottleneck_securities.assign(
        median_capacity_millions=lambda df: df["median_capacity_usd"] / 1_000_000,
        minimum_capacity_millions=lambda df: df["minimum_capacity_usd"] / 1_000_000,
    )[
        [
            "portfolio",
            "rebalance_frequency",
            "bottleneck_ticker",
            "bottleneck_observations",
            "bottleneck_fraction",
            "median_capacity_millions",
            "minimum_capacity_millions",
        ]
    ].round(
        4
    )
)

,portfolio,rebalance_frequency,bottleneck_ticker,bottleneck_observations,bottleneck_fraction,median_capacity_millions,minimum_capacity_millions
10,Composite Score,21,BNY,259,0.0983,45.8188,22.2247
45,Composite Score,21,SPG,216,0.0820,50.3406,33.0377
21,Composite Score,21,GD,196,0.0744,47.8505,30.9880
18,Composite Score,21,DUK,189,0.0717,59.9403,37.3906
44,Composite Score,21,SO,177,0.0672,56.0834,34.8843
13,Composite Score,21,COF,150,0.0569,42.8739,29.3333
19,Composite Score,21,EMR,139,0.0528,43.9809,31.4378
11,Composite Score,21,CL,125,0.0474,59.0599,36.2665
28,Composite Score,21,LIN,122,0.0463,28.3468,21.2512
46,Composite Score,21,TMUS,102,0.0387,39.8379,21.7249


#### Inspect the worst individual events

In [43]:
bottleneck_trade_detail = capacity_diagnostic.rename(
    columns={
        "bottleneck_ticker": "ticker",
    }
).merge(
    trade_liquidity[
        [
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
            "date",
            "ticker",
            "absolute_trade_weight",
            "lagged_adv_21",
        ]
    ],
    on=[
        "portfolio",
        "rebalance_frequency",
        "rebalance_offset",
        "date",
        "ticker",
    ],
    how="left",
    validate="one_to_one",
)

worst_capacity_events = (
    bottleneck_trade_detail.sort_values(
        [
            "portfolio",
            "rebalance_frequency",
            "capacity_usd",
        ]
    )
    .groupby(
        [
            "portfolio",
            "rebalance_frequency",
        ],
        group_keys=False,
    )
    .head(10)
    .assign(
        capacity_millions=lambda df: df["capacity_usd"] / 1_000_000,
        lagged_adv_21_millions=lambda df: df["lagged_adv_21"] / 1_000_000,
    )
)

display(
    worst_capacity_events[
        [
            "portfolio",
            "rebalance_frequency",
            "rebalance_offset",
            "date",
            "ticker",
            "absolute_trade_weight",
            "lagged_adv_21_millions",
            "capacity_millions",
        ]
    ].round(4)
)

C:\Users\39521\AppData\Local\Temp\ipykernel_17672\600394439.py:62: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].round(4)


,portfolio,rebalance_frequency,rebalance_offset,date,ticker,absolute_trade_weight,lagged_adv_21_millions,capacity_millions
375,Composite Score,21,3,2016-01-07,AMD,0.0500,39.4570,7.8914
2510,Composite Score,21,20,2016-02-02,AMD,0.0500,40.1260,8.0252
2385,Composite Score,21,19,2016-02-01,AMD,0.0500,40.6533,8.1307
501,Composite Score,21,4,2016-01-08,AMD,0.0500,40.9147,8.1829
2260,Composite Score,21,18,2016-01-29,AMD,0.0500,41.5822,8.3164
627,Composite Score,21,5,2016-01-11,AMD,0.0500,43.1269,8.6254
2135,Composite Score,21,17,2016-01-28,AMD,0.0500,43.1701,8.6340
2010,Composite Score,21,16,2016-01-27,AMD,0.0500,43.7247,8.7449
753,Composite Score,21,6,2016-01-12,AMD,0.0500,44.3395,8.8679
879,Composite Score,21,7,2016-01-13,AMD,0.0500,44.8617,8.9723


### Capacity bottleneck diagnostics

The bottleneck analysis supports the use of fifth-percentile capacity as the primary implementation statistic while explaining the much lower absolute historical minima.

#### Recurring bottlenecks

No individual security dominates the capacity constraint. The most frequent bottleneck represents:

- 9.83% of observations for Composite Score, attributable to BNY;
- 10.51% for Fixed 50/50, attributable to GD; and
- 9.15% for Pure Inverse Volatility, also attributable to GD.

A similar group of securities appears repeatedly across the three portfolios, including GD, SPG, SO, DUK, BNY, COF, EMR, CL and TMUS. This consistency suggests that the recurring capacity constraints reflect genuine interactions between portfolio trades and cross-sectional liquidity rather than a single anomalous security.

Nevertheless, bottleneck frequency alone does not imply severe illiquidity. Conditional median capacity for these recurring names is generally approximately $28–110 million at the 1% participation limit. A security can frequently be the least liquid trade in a rebalance without producing an exceptionally low absolute capacity.

#### Extreme events

All ten worst events for every portfolio are attributable to AMD during early 2016.

These events combine comparatively low lagged dollar volume with large security-level portfolio adjustments. The most severe cases are:

- Composite Score: a 5.00% trade against approximately $39.46 million of lagged ADV, producing capacity of $7.89 million;
- Fixed 50/50: a 5.00% trade against approximately $27.95 million of lagged ADV, producing capacity of $5.59 million; and
- Pure Inverse Volatility: a 5.13% trade against approximately $27.95 million of lagged ADV, producing capacity of $5.44 million.

The calculations are internally consistent with the 1% participation constraint. The low historical minima are therefore not numerical errors: they result from attempting to execute a large portfolio weight change in a security whose historical dollar volume was much lower than its modern liquidity.

Several Composite events represent the same early-2016 liquidity environment observed under different rebalance offsets. They should therefore be interpreted as related phase scenarios rather than independent stress events.

The slight excess over 5% in some Pure Inverse Volatility trades is consistent with its sleeve scaling and gross exposure exceeding one; the trade-weight cap, if any, is not necessarily equivalent to a final portfolio-level 5% order cap.

#### Interpretation

The recurring bottleneck results show that capacity risk is distributed across several securities. The extreme lower tail, however, is concentrated in a single early-sample AMD episode.

This distinction explains why the absolute historical minima are much lower than the fifth-percentile capacity estimates. The fifth percentile describes recurring adverse implementation conditions, whereas the minimum captures a specific historical stress case.

The early AMD events should remain in the reported stress diagnostics, but they should not define the principal capacity estimate. This is particularly important because the research universe uses a static set of present-day S&P 100 constituents. Historical liquidity for a current constituent may not represent the investability conditions that a point-in-time large-cap universe would have imposed at that date.

The primary 1% participation estimates therefore remain approximately:

- $25 million for Composite Score;
- $40 million for Fixed 50/50 Sleeves; and
- $38 million for Pure Inverse Volatility.

The next test compares capacity across subperiods to determine how much these full-sample estimates are influenced by the earlier, less-liquid part of the sample.

### Capacity by subperiods

In [44]:
CAPACITY_SUBPERIODS = {
    "2016–2018": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2018-12-31"),
    ),
    "2019–2022": (
        pd.Timestamp("2019-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "2023–present": (
        pd.Timestamp("2023-01-01"),
        capacity_diagnostic["date"].max(),
    ),
}

capacity_subperiod_parts = []

for subperiod_name, (
    start_date,
    end_date,
) in CAPACITY_SUBPERIODS.items():
    subperiod_data = capacity_diagnostic.loc[
        capacity_diagnostic["date"].between(
            start_date,
            end_date,
        )
    ].copy()

    subperiod_data["subperiod"] = subperiod_name

    capacity_subperiod_parts.append(subperiod_data)

capacity_subperiod_daily = pd.concat(
    capacity_subperiod_parts,
    ignore_index=True,
)

In [45]:
capacity_subperiod_by_offset = (
    capacity_subperiod_daily.groupby(
        [
            "portfolio",
            "rebalance_frequency",
            "subperiod",
            "rebalance_offset",
        ],
        sort=True,
    )
    .agg(
        rebalance_observations=("date", "size"),
        fifth_percentile_capacity_usd=(
            "capacity_usd",
            lambda values: values.quantile(0.05),
        ),
        tenth_percentile_capacity_usd=(
            "capacity_usd",
            lambda values: values.quantile(0.10),
        ),
        median_capacity_usd=(
            "capacity_usd",
            "median",
        ),
        minimum_capacity_usd=(
            "capacity_usd",
            "min",
        ),
    )
    .reset_index()
)

In [46]:
capacity_subperiod_summary = (
    capacity_subperiod_by_offset.groupby(
        [
            "portfolio",
            "rebalance_frequency",
            "subperiod",
        ],
        sort=False,
    )
    .agg(
        offset_count=(
            "rebalance_offset",
            "nunique",
        ),
        minimum_phase_observations=(
            "rebalance_observations",
            "min",
        ),
        median_fifth_percentile_capacity_usd=(
            "fifth_percentile_capacity_usd",
            "median",
        ),
        worst_phase_fifth_percentile_capacity_usd=(
            "fifth_percentile_capacity_usd",
            "min",
        ),
        median_tenth_percentile_capacity_usd=(
            "tenth_percentile_capacity_usd",
            "median",
        ),
        worst_phase_tenth_percentile_capacity_usd=(
            "tenth_percentile_capacity_usd",
            "min",
        ),
        median_capacity_usd=(
            "median_capacity_usd",
            "median",
        ),
        worst_historical_capacity_usd=(
            "minimum_capacity_usd",
            "min",
        ),
    )
    .reset_index()
)

In [47]:
capacity_subperiod_summary_millions = capacity_subperiod_summary.copy()

subperiod_capacity_columns = [
    column
    for column in capacity_subperiod_summary_millions.columns
    if column.endswith("_capacity_usd")
]

capacity_subperiod_summary_millions[subperiod_capacity_columns] = (
    capacity_subperiod_summary_millions[subperiod_capacity_columns] / 1_000_000.0
)

display(capacity_subperiod_summary_millions.round(2))

,portfolio,rebalance_frequency,subperiod,offset_count,minimum_phase_observations,median_fifth_percentile_capacity_usd,worst_phase_fifth_percentile_capacity_usd,median_tenth_percentile_capacity_usd,worst_phase_tenth_percentile_capacity_usd,median_capacity_usd,worst_historical_capacity_usd
0,Composite Score,21,2016–2018,21,35,22.31,19.23,25.69,23.07,39.43,7.89
1,Composite Score,21,2019–2022,21,48,33.79,28.35,35.12,33.84,49.19,21.72
2,Composite Score,21,2023–present,21,41,47.30,38.68,50.57,46.41,73.61,33.04
3,Fixed 50/50 Sleeves,10,2016–2018,10,75,33.33,28.71,38.41,33.84,68.17,5.59
4,Fixed 50/50 Sleeves,10,2019–2022,10,100,63.93,53.38,71.99,65.06,96.86,38.70
5,Fixed 50/50 Sleeves,10,2023–present,10,87,78.13,72.02,84.74,83.02,122.99,41.44
6,Pure Inverse Volatility,10,2016–2018,10,75,31.54,29.96,35.44,32.04,65.94,5.44
7,Pure Inverse Volatility,10,2019–2022,10,100,57.27,52.52,64.18,59.00,89.54,38.15
8,Pure Inverse Volatility,10,2023–present,10,87,74.86,72.10,84.29,76.61,123.46,40.55


### Capacity stability across subperiods

Capacity increases materially across the sample for all three implementations.

Using the conservative 1% participation limit, worst-phase fifth-percentile capacity rises from:

- $19.23 million in 2016–2018 to $38.68 million in 2023–present for Composite Score;
- $28.71 million to $72.02 million for Fixed 50/50 Sleeves; and
- $29.96 million to $72.10 million for Pure Inverse Volatility.

Relative to the early subperiod, recent capacity is approximately 101% higher for Composite Score, 151% higher for Fixed 50/50, and 141% higher for Pure Inverse Volatility.

The increase is not confined to a single tail statistic. The fifth percentile, tenth percentile, median, and historical minimum all improve substantially through time. This indicates a broad shift in implementation capacity rather than a result driven solely by the disappearance of the early-2016 AMD stress events.

In 2023–present, the worst capacity observed across all phase implementations is:

- $33.04 million for Composite Score;
- $41.44 million for Fixed 50/50; and
- $40.55 million for Pure Inverse Volatility.

Thus, even the recent-period historical minima are much stronger than the severe early-sample stress values of approximately $5–8 million.

The relative ranking is stable. Both 10-day sleeve implementations provide substantially greater same-day capacity than the 21-day Composite implementation. Their recent worst-phase fifth-percentile capacities are approximately $72 million, around 86% above Composite's $38.68 million.

Fixed 50/50 and Pure Inverse Volatility have nearly indistinguishable recent capacity. Their small capacity differences are not sufficiently large or stable to influence portfolio selection materially.

#### Reporting interpretation

Two capacity estimates should be retained:

- **Full-history conservative capacity:** approximately $25 million for Composite, $40 million for Fixed 50/50, and $38 million for Pure Inverse Volatility.
- **Recent-period implementation capacity:** approximately $39 million for Composite and $72 million for both sleeve portfolios.

The full-history estimate reflects adverse conditions across the complete backtest and is appropriate for conservative reporting. The recent-period estimate is more informative about present-day scalability because dollar trading volume has increased substantially over the sample.

These figures remain nominal-dollar, liquidity-screening estimates. The historical increase may reflect market growth, inflation, changing index membership and company-specific liquidity development. The static present-day S&P 100 universe also limits the interpretation of early-period capacity.

The capacity analysis therefore supports the implementation feasibility of all three portfolios at moderate AUM, while showing a clear scalability advantage for the more frequent 10-day sleeve implementations.

## Notebook 06 summary and implementation decisions

This notebook tested implementation robustness across rebalance frequencies, calendar offsets, transaction-cost assumptions, rolling periods, turnover concentration and security-level liquidity.

The selected implementation frequencies are 21 trading days for Composite Score and 10 trading days for the two sleeve portfolios. The main conclusions are not dependent on a favourable rebalance offset. Composite retains the strongest combination of net performance and low cumulative turnover, while the more frequent sleeve implementations generally make smaller same-day adjustments and therefore provide greater capacity.

At a conservative 1% ADV participation limit, full-history worst-phase fifth-percentile capacity is approximately:

- $24.77 million for Composite Score;
- $40.17 million for Fixed 50/50 Sleeves; and
- $38.34 million for Pure Inverse Volatility.

For 2023–present, the corresponding estimates rise to $38.68 million, $72.02 million and $72.10 million. Both full-history and recent-period estimates will be retained: the former as a conservative historical measure and the latter as a more relevant indication of current scalability.

Recurring bottlenecks are distributed across several securities. The absolute historical minima—$7.89 million, $5.59 million and $5.44 million—are all driven by AMD liquidity events in early 2016 and are retained as stress scenarios rather than primary capacity estimates. Interpretation remains limited by the static present-day universe, nominal-dollar ADV and one-day execution assumption.

Composite Score is retained as the primary portfolio for moderate AUM, provided expected capital remains comfortably below its conservative capacity estimate. Fixed 50/50 and Pure Inverse Volatility remain useful implementation and higher-capacity benchmarks. Further capacity modelling is optional; the project will now move to performance and risk attribution, followed by monitoring and final documentation.

#### Imports and frozen specifications

In [60]:
from alpha_research.artifacts import (
    write_attribution_artifacts,
)
from alpha_research.attribution import (
    reconcile_security_attribution,
)
from alpha_research.config.research import (
    selected_implementations_frame,
)
from alpha_research.workflows import (
    build_frozen_strategy_target_weights,
    build_selected_attribution_datasets,
)

SELECTED_IMPLEMENTATIONS = selected_implementations_frame()

display(SELECTED_IMPLEMENTATIONS)

,portfolio,rebalance_frequency,rebalance_offset,role
0,Composite Score,21,0,Primary specification
1,Fixed 50/50 Sleeves,10,0,Transparent sleeve benchmark
2,Pure Inverse Volatility,10,0,Risk-based sleeve benchmark


#### Construct the complete handoff

In [64]:
SELECTION_KEY_COLUMNS = [
    "portfolio",
    "rebalance_frequency",
    "rebalance_offset",
]

selected_portfolio_daily_reference = (
    rebalance_offset_daily.merge(
        SELECTED_IMPLEMENTATIONS,
        on=SELECTION_KEY_COLUMNS,
        how="inner",
        validate="many_to_one",
    )
    .sort_values(["portfolio", "date"])
    .reset_index(drop=True)
)

selected_benchmark_daily = (
    spy_daily.loc[
        spy_daily["date"].isin(robustness_reference_dates),
        ["date", "gross_return"],
    ]
    .rename(
        columns={
            "gross_return": "benchmark_return",
        }
    )
    .assign(benchmark="SPY")[["date", "benchmark", "benchmark_return"]]
    .sort_values("date")
    .reset_index(drop=True)
)

selected_target_weights_by_portfolio = build_frozen_strategy_target_weights(panel)

attribution_datasets = build_selected_attribution_datasets(
    return_panel=return_panel,
    target_weights_by_portfolio=(selected_target_weights_by_portfolio),
    benchmark_daily=selected_benchmark_daily,
    analysis_dates=robustness_reference_dates,
    selected_implementations=(SELECTED_IMPLEMENTATIONS),
)

selected_portfolio_daily = attribution_datasets["portfolio_daily"]
selected_security_holdings = attribution_datasets["security_holdings"]
selected_target_weights = attribution_datasets["target_weights"]
selected_security_daily = attribution_datasets["security_daily"]

#### Replay and attribution audits

In [65]:
reference_portfolio_daily = selected_portfolio_daily_reference.sort_values(
    ["portfolio", "date"]
).reset_index(drop=True)

replayed_portfolio_daily = selected_portfolio_daily.sort_values(
    ["portfolio", "date"]
).reset_index(drop=True)

column_sets_match = set(reference_portfolio_daily.columns) == set(
    replayed_portfolio_daily.columns
)

portfolio_keys_match = reference_portfolio_daily[["portfolio", "date"]].equals(
    replayed_portfolio_daily[["portfolio", "date"]]
)

portfolio_values_match = False

if column_sets_match:
    try:
        pd.testing.assert_frame_equal(
            replayed_portfolio_daily[reference_portfolio_daily.columns],
            reference_portfolio_daily,
            check_dtype=False,
            check_categorical=False,
            check_exact=False,
            rtol=0.0,
            atol=1e-12,
        )
        portfolio_values_match = True
    except AssertionError as error:
        print("Selected portfolio replay mismatch:")
        print(error)

selected_replay_audit = pd.DataFrame(
    [
        {
            "replayed_rows": len(replayed_portfolio_daily),
            "reference_rows": len(reference_portfolio_daily),
            "column_sets_match": column_sets_match,
            "keys_match": portfolio_keys_match,
            "values_match": portfolio_values_match,
            "audit_passes": (
                column_sets_match and portfolio_keys_match and portfolio_values_match
            ),
        }
    ]
)

security_attribution_audit = reconcile_security_attribution(
    selected_portfolio_daily,
    selected_security_daily,
).set_index("portfolio")

handoff_schema = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "rows": len(dataset),
            "columns": len(dataset.columns),
        }
        for dataset_name, dataset in attribution_datasets.items()
    ]
)

display(selected_replay_audit)
display(security_attribution_audit)
display(handoff_schema)

if not selected_replay_audit["audit_passes"].all():
    raise ValueError("Selected portfolio replay failed.")

if not security_attribution_audit["audit_passes"].all():
    raise ValueError("Security attribution failed.")

,replayed_rows,reference_rows,column_sets_match,keys_match,values_match,audit_passes
0,7905,7905,True,True,True,True


,observations,max_abs_long_return_difference,max_abs_short_return_difference,max_abs_gross_return_difference,max_abs_turnover_difference,max_abs_transaction_cost_difference,max_abs_net_return_difference,max_abs_long_exposure_difference,max_abs_short_exposure_difference,max_abs_missing_return_weight_difference,maximum_absolute_difference,audit_passes
portfolio,,,,,,,,,,,,
Composite Score,2635,1.387779e-17,1.387779e-17,1.387779e-17,2.220446e-16,4.336809e-19,1.387779e-17,2.220446e-16,4.440892e-16,0.0,4.440892e-16,True
Fixed 50/50 Sleeves,2635,1.387779e-17,1.387779e-17,2.081668e-17,2.220446e-16,3.252607e-19,2.081668e-17,2.220446e-16,2.220446e-16,0.0,2.220446e-16,True
Pure Inverse Volatility,2635,1.387779e-17,1.387779e-17,2.081668e-17,4.440892e-16,4.336809e-19,2.081668e-17,2.220446e-16,2.220446e-16,0.0,4.440892e-16,True


,dataset,rows,columns
0,selected_implementations,3,4
1,portfolio_daily,7905,20
2,security_holdings,778239,9
3,target_weights,64088,7
4,benchmark_daily,2635,3
5,security_daily,407676,23


#### Validated exports

In [66]:
attribution_export_manifest = write_attribution_artifacts(
    attribution_datasets,
    PROCESSED_DATA_DIR,
)

display(attribution_export_manifest)

print(
    "All attribution read-back checks pass:",
    attribution_export_manifest["read_back_passes"].all(),
)

,dataset,file,rows,columns,start_date,end_date,read_back_passes
0,selected_implementations,attribution_selected_implementations.parquet,3,4,NaT,NaT,True
1,portfolio_daily,attribution_portfolio_daily.parquet,7905,20,2016-01-07,2026-07-01,True
2,security_holdings,attribution_security_holdings.parquet,778239,9,2016-01-07,2026-07-01,True
3,target_weights,attribution_target_weights.parquet,64088,7,2016-01-14,2026-06-17,True
4,benchmark_daily,attribution_benchmark_daily.parquet,2635,3,2016-01-07,2026-07-01,True
5,security_daily,attribution_security_daily.parquet,407676,23,2016-01-07,2026-07-01,True


All attribution read-back checks pass: True
